In [28]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
IMAGE_DIR = EXPERIMENT_DIR / "okra_images_final"
MANIFEST_PATH = EXPERIMENT_DIR / "okra_final_2799_manifest_drive_paths.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Saved manifest not found: {MANIFEST_PATH}")
if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"Saved image folder not found: {IMAGE_DIR}")

okra_df = pd.read_csv(MANIFEST_PATH)
missing = [p for p in okra_df["image_path"].astype(str) if not Path(p).is_file()]

print("Manifest rows:", len(okra_df))
print("Images missing from Drive:", len(missing))
print("\nImages by class:")
print(okra_df["label"].value_counts().sort_index().to_string())

if missing:
    print("\nMissing path examples:")
    print("\n".join(missing[:10]))
    raise RuntimeError("Some saved images are missing; stop here and report the output.")

print("\nDrive copy is ready to continue.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Manifest rows: 2799
Images missing from Drive: 0

Images by class:
label
Healthy_Leaf                693
Insects_Eaten_Leaf          575
Leaf_Curl                   343
Leaf_Spot                   302
Yellow_Vein_Mosaic_Virus    886

Drive copy is ready to continue.


In [2]:
from pathlib import Path

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)

for path in sorted(EXPERIMENT_DIR.glob("*.csv")):
    print(path.name)

okra_additional_images_duplicate_audit.csv
okra_candidates_distance_2_50.csv
okra_clean_manifest.csv
okra_combined_exact_duplicate_report.csv
okra_combined_initial_manifest.csv
okra_combined_manifest_with_sha256.csv
okra_exact_duplicate_report.csv
okra_exact_duplicates_set_aside.csv
okra_final_2799_manifest_drive_paths.csv
okra_five_class_near_duplicate_candidates.csv
okra_five_class_near_duplicate_candidates_256bit.csv
okra_five_class_perceptual_hashes.csv
okra_five_class_perceptual_hashes_256bit.csv
okra_fruit_rot_class_set_aside.csv
okra_higher_distance_candidates_50_80.csv
okra_image_clarity_audit.csv
okra_initial_manifest.csv
okra_manifest_after_confirmed_50_80_pairs.csv
okra_manifest_after_distance_2_50_groups.csv
okra_manifest_after_near_duplicate_batch_0_29.csv
okra_manifest_after_remaining_30_groups.csv
okra_manifest_with_sha256.csv
okra_manifest_without_fruit_rot.csv
okra_near_duplicate_candidates.csv
okra_near_duplicate_review_0_29.csv
okra_near_duplicate_review_256bit_0_29.

In [3]:
from pathlib import Path
from collections import defaultdict
import json

import pandas as pd
from sklearn.model_selection import train_test_split

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
MANIFEST_PATH = EXPERIMENT_DIR / "okra_final_2799_manifest_drive_paths.csv"

SPLIT_SEED = 42
OUTPUT_NAMES = {
    "train": "okra_train_manifest.csv",
    "validation": "okra_validation_manifest.csv",
    "internal_test": "okra_internal_test_manifest.csv",
}

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Final Drive manifest not found: {MANIFEST_PATH}")

df = pd.read_csv(MANIFEST_PATH)
required = {"image_path", "label"}
if not required.issubset(df.columns):
    raise ValueError(f"Manifest must contain {sorted(required)}")

df["image_path"] = df["image_path"].astype(str)
if "original_colab_path" in df.columns:
    df["group_path"] = df["original_colab_path"].fillna(df["image_path"]).astype(str)
else:
    df["group_path"] = df["image_path"]

# Reconstruct confirmed duplicate relationships from the removal logs.
parent = {}

def find(path):
    parent.setdefault(path, path)
    if parent[path] != path:
        parent[path] = find(parent[path])
    return parent[path]

def union(a, b):
    root_a, root_b = find(a), find(b)
    if root_a != root_b:
        parent[root_b] = root_a

log_files = [
    EXPERIMENT_DIR / "okra_near_duplicates_set_aside_batch_0_29.csv",
    EXPERIMENT_DIR / "okra_remaining_30_groups_set_aside.csv",
    EXPERIMENT_DIR / "okra_set_aside_confirmed_50_80_pairs.csv",
    EXPERIMENT_DIR / "okra_set_aside_distance_2_50_groups.csv",
]

logs_read = []
for log_path in log_files:
    if not log_path.exists():
        continue

    log = pd.read_csv(log_path)
    kept_col = next(
        (c for c in ["kept_image_path", "kept_path"] if c in log.columns), None
    )
    removed_col = next(
        (c for c in ["set_aside_image_path", "removed_image_path", "image_path"]
         if c in log.columns and c != kept_col),
        None,
    )
    if kept_col is None or removed_col is None:
        print(f"Skipping unrecognized log format: {log_path.name} ({log.columns.tolist()})")
        continue

    for kept, removed in log[[kept_col, removed_col]].dropna().itertuples(index=False, name=None):
        union(str(kept), str(removed))
    logs_read.append(log_path.name)

# Assign a component ID to each retained image, using its original Colab path.
df["duplicate_group"] = df["group_path"].map(lambda p: find(str(p)))

# Do not silently proceed if two retained images still belong to one confirmed group.
group_sizes = df.groupby("duplicate_group").size()
multi_image_groups = group_sizes[group_sizes > 1]
if not multi_image_groups.empty:
    examples = df[df["duplicate_group"].isin(multi_image_groups.index)][
        ["image_path", "label", "duplicate_group"]
    ].head(12)
    display(examples)
    raise RuntimeError(
        f"{len(multi_image_groups)} duplicate group(s) still contain multiple retained "
        "images. Resolve these before splitting to prevent leakage."
    )

# Stratified split is safe here because every surviving duplicate component is a singleton.
train_val, test = train_test_split(
    df,
    test_size=0.15,
    random_state=SPLIT_SEED,
    stratify=df["label"],
)
train, validation = train_test_split(
    train_val,
    test_size=0.15 / 0.85,
    random_state=SPLIT_SEED,
    stratify=train_val["label"],
)

splits = {
    "train": train.reset_index(drop=True),
    "validation": validation.reset_index(drop=True),
    "internal_test": test.reset_index(drop=True),
}

for name, part in splits.items():
    part.drop(columns=["group_path"], errors="ignore").to_csv(
        EXPERIMENT_DIR / OUTPUT_NAMES[name],
        index=False,
    )

# Confirm no image or duplicate group crosses splits.
for a, b in [("train", "validation"), ("train", "internal_test"),
             ("validation", "internal_test")]:
    assert set(splits[a]["image_path"]).isdisjoint(set(splits[b]["image_path"]))
    assert set(splits[a]["duplicate_group"]).isdisjoint(set(splits[b]["duplicate_group"]))

print("Duplicate-removal logs used:", logs_read)
print("Final images:", len(df))
print("Images per split:")
for name, part in splits.items():
    print(f"  {name}: {len(part)} ({len(part) / len(df):.1%})")

print("\nClass counts by split:")
display(pd.crosstab(
    pd.concat([
        part.assign(split=name)
        for name, part in splits.items()
    ])["split"],
    pd.concat([
        part.assign(split=name)
        for name, part in splits.items()
    ])["label"],
))

config = {
    "dataset": "Okra five-class dataset",
    "source_manifest": str(MANIFEST_PATH),
    "classes": sorted(df["label"].unique().tolist()),
    "total_images": len(df),
    "split_ratios_target": {"train": 0.70, "validation": 0.15, "internal_test": 0.15},
    "random_state": SPLIT_SEED,
    "split_counts": {name: len(part) for name, part in splits.items()},
    "duplicate_logs_used": logs_read,
    "duplicate_group_cross_split_check": "passed",
}
config_path = EXPERIMENT_DIR / "okra_split_config.json"
config_path.write_text(json.dumps(config, indent=2))

print("\nSaved split manifests and config in:", EXPERIMENT_DIR)

Skipping unrecognized log format: okra_near_duplicates_set_aside_batch_0_29.csv (['image_path', 'label', 'width', 'height', 'format', 'dataset_source', 'sha256', 'review_group', 'kept_representative_path', 'set_aside_reason'])
Duplicate-removal logs used: ['okra_remaining_30_groups_set_aside.csv', 'okra_set_aside_confirmed_50_80_pairs.csv', 'okra_set_aside_distance_2_50_groups.csv']
Final images: 2799
Images per split:
  train: 1959 (70.0%)
  validation: 420 (15.0%)
  internal_test: 420 (15.0%)

Class counts by split:


ValueError: cannot reindex on an axis with duplicate labels

In [4]:
from pathlib import Path
from collections import defaultdict
import json

import pandas as pd
from sklearn.model_selection import train_test_split

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
MANIFEST_PATH = EXPERIMENT_DIR / "okra_final_2799_manifest_drive_paths.csv"

SPLIT_SEED = 42
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Final manifest not found: {MANIFEST_PATH}")

df = pd.read_csv(MANIFEST_PATH)
if not {"image_path", "label"}.issubset(df.columns):
    raise ValueError("Final manifest must contain image_path and label.")

df["image_path"] = df["image_path"].astype(str)
df["group_path"] = (
    df["original_colab_path"].fillna(df["image_path"]).astype(str)
    if "original_colab_path" in df.columns
    else df["image_path"]
)

# Reconstruct duplicate links from every applicable set-aside log.
parent = {}

def find(path):
    parent.setdefault(path, path)
    if parent[path] != path:
        parent[path] = find(parent[path])
    return parent[path]

def union(a, b):
    root_a, root_b = find(a), find(b)
    if root_a != root_b:
        parent[root_b] = root_a

log_specs = [
    # This first log has a different schema from the later logs.
    ("okra_near_duplicates_set_aside_batch_0_29.csv",
     "kept_representative_path", "image_path"),
    ("okra_remaining_30_groups_set_aside.csv",
     "kept_image_path", "set_aside_image_path"),
    ("okra_set_aside_confirmed_50_80_pairs.csv",
     "kept_image_path", "set_aside_image_path"),
    ("okra_set_aside_distance_2_50_groups.csv",
     "kept_image_path", "set_aside_image_path"),
]

logs_used = []
for filename, keep_col, removed_col in log_specs:
    log_path = EXPERIMENT_DIR / filename
    if not log_path.exists():
        continue

    log = pd.read_csv(log_path)
    if keep_col not in log.columns or removed_col not in log.columns:
        print(f"Skipping {filename}: required columns not found.")
        continue

    for kept, removed in log[[keep_col, removed_col]].dropna().itertuples(
        index=False, name=None
    ):
        union(str(kept), str(removed))
    logs_used.append(filename)

df["duplicate_group"] = df["group_path"].map(lambda path: find(str(path)))

# A retained duplicate group must not be split across train/validation/test.
multi_groups = df.groupby("duplicate_group").size()
multi_groups = multi_groups[multi_groups > 1]
if not multi_groups.empty:
    display(
        df[df["duplicate_group"].isin(multi_groups.index)][
            ["image_path", "label", "duplicate_group"]
        ].head(20)
    )
    raise RuntimeError(
        f"{len(multi_groups)} confirmed duplicate group(s) still have multiple "
        "retained images. Resolve before splitting."
    )

# All retained duplicate groups are singletons, so stratified splits are safe.
train_val, test = train_test_split(
    df,
    test_size=0.15,
    random_state=SPLIT_SEED,
    stratify=df["label"],
)
train, validation = train_test_split(
    train_val,
    test_size=0.15 / 0.85,
    random_state=SPLIT_SEED,
    stratify=train_val["label"],
)

splits = {
    "train": train.reset_index(drop=True),
    "validation": validation.reset_index(drop=True),
    "internal_test": test.reset_index(drop=True),
}
output_names = {
    "train": "okra_train_manifest.csv",
    "validation": "okra_validation_manifest.csv",
    "internal_test": "okra_internal_test_manifest.csv",
}

for name, part in splits.items():
    part.drop(columns=["group_path"], errors="ignore").to_csv(
        EXPERIMENT_DIR / output_names[name], index=False
    )

# Verify no image or duplicate component crosses split boundaries.
for left, right in [
    ("train", "validation"),
    ("train", "internal_test"),
    ("validation", "internal_test"),
]:
    assert set(splits[left]["image_path"]).isdisjoint(splits[right]["image_path"])
    assert set(splits[left]["duplicate_group"]).isdisjoint(
        splits[right]["duplicate_group"]
    )

combined = pd.concat(
    [part.assign(split=name) for name, part in splits.items()],
    ignore_index=True,
)
class_counts = pd.crosstab(combined["split"], combined["label"])

print("Duplicate-removal logs used:", logs_used)
print("Final images:", len(df))
print("Split totals:")
for name, part in splits.items():
    print(f"  {name}: {len(part)} ({len(part) / len(df):.1%})")
print("\nClass counts by split:")
display(class_counts)

config = {
    "dataset": "Okra five-class dataset",
    "source_manifest": str(MANIFEST_PATH),
    "classes": sorted(df["label"].unique().tolist()),
    "total_images": len(df),
    "target_split_ratios": {
        "train": 0.70,
        "validation": 0.15,
        "internal_test": 0.15,
    },
    "random_state": SPLIT_SEED,
    "split_counts": {name: len(part) for name, part in splits.items()},
    "duplicate_logs_used": logs_used,
    "duplicate_group_cross_split_check": "passed",
}
(EXPERIMENT_DIR / "okra_split_config.json").write_text(
    json.dumps(config, indent=2)
)

print("\nSaved split manifests and config to:", EXPERIMENT_DIR)

Duplicate-removal logs used: ['okra_near_duplicates_set_aside_batch_0_29.csv', 'okra_remaining_30_groups_set_aside.csv', 'okra_set_aside_confirmed_50_80_pairs.csv', 'okra_set_aside_distance_2_50_groups.csv']
Final images: 2799
Split totals:
  train: 1959 (70.0%)
  validation: 420 (15.0%)
  internal_test: 420 (15.0%)

Class counts by split:


label,Healthy_Leaf,Insects_Eaten_Leaf,Leaf_Curl,Leaf_Spot,Yellow_Vein_Mosaic_Virus
split,,,,,
internal_test,104,86,52,45,133
train,485,403,239,212,620
validation,104,86,52,45,133



Saved split manifests and config to: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment


In [6]:
from pathlib import Path
import pandas as pd
from PIL import Image, ImageOps
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
SOURCE_MANIFEST = EXPERIMENT_DIR / "okra_final_2799_manifest_drive_paths.csv"
CACHE_DIR = Path("/content/okra_cache_448")
CACHE_MANIFEST = EXPERIMENT_DIR / "okra_resized_cache_manifest.csv"

IMAGE_SIZE = 448
FILL_COLOR = (127, 127, 127)

df = pd.read_csv(SOURCE_MANIFEST)
if not {"image_path", "label"}.issubset(df.columns):
    raise ValueError("Manifest must contain image_path and label.")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
cached_paths = []
errors = []

for row in tqdm(df.itertuples(index=False), total=len(df), desc="Caching Okra images"):
    source_path = Path(str(row.image_path))
    label = str(row.label)
    target_path = CACHE_DIR / label / f"{source_path.stem}.jpg"
    target_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        if not source_path.is_file():
            raise FileNotFoundError(source_path)

        if not target_path.exists():
            with Image.open(source_path) as source:
                image = ImageOps.exif_transpose(source).convert("RGB")
                image = ImageOps.pad(
                    image,
                    (IMAGE_SIZE, IMAGE_SIZE),
                    method=Image.Resampling.LANCZOS,
                    color=FILL_COLOR,
                )
                image.save(target_path, format="JPEG", quality=95)

        cached_paths.append(str(target_path))
    except Exception as exc:
        cached_paths.append(None)
        errors.append({
            "source_path": str(source_path),
            "error": str(exc),
        })

if errors:
    print("Cache errors:", len(errors))
    display(pd.DataFrame(errors).head(20))
    raise RuntimeError("Fix cache errors before proceeding.")

cache_df = df.copy()
cache_df["source_image_path"] = cache_df["image_path"].astype(str)
cache_df["cached_image_path"] = cached_paths
cache_df.to_csv(CACHE_MANIFEST, index=False)

missing = [path for path in cached_paths if not Path(path).is_file()]
if missing:
    raise RuntimeError(f"{len(missing)} cached image files are missing.")

print("Images cached:", len(cache_df))
print("Cache folder:", CACHE_DIR)
print("Cache manifest:", CACHE_MANIFEST)
print("Example cached image:", cached_paths[0])






Caching Okra images:   0%|          | 0/2799 [00:00<?, ?it/s]

Images cached: 2799
Cache folder: /content/okra_cache_448
Cache manifest: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_resized_cache_manifest.csv
Example cached image: /content/okra_cache_448/Leaf_Spot/LS-00009.jpg


In [7]:
from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import functional as TF
from IPython.display import display

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
CACHE_MANIFEST_PATH = EXPERIMENT_DIR / "okra_resized_cache_manifest.csv"

IMAGE_SIZE = 448
TILE_SIZE = 288
BATCH_SIZE = 4
NUM_WORKERS = 0  # Reliable in Colab; increase later if loading is slow.

CLASS_NAMES = [
    "Healthy_Leaf",
    "Insects_Eaten_Leaf",
    "Leaf_Curl",
    "Leaf_Spot",
    "Yellow_Vein_Mosaic_Virus",
]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

cache_df = pd.read_csv(CACHE_MANIFEST_PATH)
cache_df["image_path"] = cache_df["source_image_path"].astype(str)
cache_df["cached_image_path"] = cache_df["cached_image_path"].astype(str)

cached_path_map = dict(zip(cache_df["image_path"], cache_df["cached_image_path"]))

def load_split(filename):
    split_df = pd.read_csv(EXPERIMENT_DIR / filename)
    split_df["image_path"] = split_df["image_path"].astype(str)
    split_df["cached_image_path"] = split_df["image_path"].map(cached_path_map)

    if split_df["cached_image_path"].isna().any():
        missing = split_df.loc[split_df["cached_image_path"].isna(), "image_path"].head()
        raise ValueError(f"Could not map split paths to cache:\n{missing.to_string(index=False)}")

    missing_files = [
        p for p in split_df["cached_image_path"]
        if not Path(p).is_file()
    ]
    if missing_files:
        raise FileNotFoundError(
            f"{len(missing_files)} cached split images are missing. "
            f"Example: {missing_files[:3]}"
        )

    unknown = sorted(set(split_df["label"]) - set(CLASS_TO_IDX))
    if unknown:
        raise ValueError(f"Unexpected labels in {filename}: {unknown}")

    return split_df

train_df = load_split("okra_train_manifest.csv")
val_df = load_split("okra_validation_manifest.csv")
test_df = load_split("okra_internal_test_manifest.csv")

# Four overlapping corner tiles from the 448x448 cached image.
tile_starts = [
    (0, 0),
    (IMAGE_SIZE - TILE_SIZE, 0),
    (0, IMAGE_SIZE - TILE_SIZE),
    (IMAGE_SIZE - TILE_SIZE, IMAGE_SIZE - TILE_SIZE),
]

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

class OkraImageDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        with Image.open(row["cached_image_path"]) as source:
            image = source.convert("RGB")

        full = TF.normalize(TF.to_tensor(image), IMAGENET_MEAN, IMAGENET_STD)

        tiles = []
        for left, top in tile_starts:
            tile = image.crop((left, top, left + TILE_SIZE, top + TILE_SIZE))
            tiles.append(
                TF.normalize(TF.to_tensor(tile), IMAGENET_MEAN, IMAGENET_STD)
            )

        label = CLASS_TO_IDX[row["label"]]
        return full, torch.stack(tiles), torch.tensor(label, dtype=torch.long)

train_loader = DataLoader(
    OkraImageDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    OkraImageDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    OkraImageDataset(test_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

full_batch, tile_batch, label_batch = next(iter(train_loader))
print("Classes:", CLASS_NAMES)
print("Train / validation / internal test:", len(train_df), len(val_df), len(test_df))
print("Full batch:", tuple(full_batch.shape))
print("Tile batch:", tuple(tile_batch.shape))
print("Label batch:", tuple(label_batch.shape))
print("GPU available:", torch.cuda.is_available())
print("Train / validation / test batches:",
      len(train_loader), len(val_loader), len(test_loader))

display(
    pd.concat(
        [
            train_df.assign(split="train"),
            val_df.assign(split="validation"),
            test_df.assign(split="internal_test"),
        ],
        ignore_index=True,
    ).groupby(["split", "label"]).size().unstack(fill_value=0)
)

Classes: ['Healthy_Leaf', 'Insects_Eaten_Leaf', 'Leaf_Curl', 'Leaf_Spot', 'Yellow_Vein_Mosaic_Virus']
Train / validation / internal test: 1959 420 420
Full batch: (4, 3, 448, 448)
Tile batch: (4, 4, 3, 288, 288)
Label batch: (4,)
GPU available: True
Train / validation / test batches: 490 105 105


label,Healthy_Leaf,Insects_Eaten_Leaf,Leaf_Curl,Leaf_Spot,Yellow_Vein_Mosaic_Virus
split,,,,,
internal_test,104,86,52,45,133
train,485,403,239,212,620
validation,104,86,52,45,133


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50, ResNet50_Weights

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

class SpatialSelfAttention(nn.Module):
    def __init__(self, channels=256, num_heads=8):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.attention = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            batch_first=True,
        )
        self.gate = nn.Conv2d(channels * 2, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, height, width = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        normalized = self.norm(tokens)
        attended, _ = self.attention(
            normalized, normalized, normalized, need_weights=False
        )
        attended = attended.transpose(1, 2).reshape(
            batch, channels, height, width
        )
        gate = torch.sigmoid(self.gate(torch.cat([x, attended], dim=1)))
        return x + gate * attended


class OkraResNet50Attention(nn.Module):
    def __init__(self, num_classes=len(CLASS_NAMES), pretrained=True):
        super().__init__()

        weights = ResNet50_Weights.DEFAULT if pretrained else None
        backbone = resnet50(weights=weights)

        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.project2 = nn.Conv2d(512, 256, kernel_size=1)
        self.project3 = nn.Conv2d(1024, 256, kernel_size=1)
        self.project4 = nn.Conv2d(2048, 256, kernel_size=1)

        self.fusion = nn.Sequential(
            nn.Conv2d(768, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.self_attention = SpatialSelfAttention(256, num_heads=8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.tile_score = nn.Linear(256, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.35),
            nn.Linear(768, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(512, num_classes),
        )

    def encode_view(self, images):
        x = self.stem(images)
        x = self.layer1(x)

        fine = self.layer2(x)
        middle = self.layer3(fine)
        deep = self.layer4(middle)

        output_size = fine.shape[-2:]
        fine = self.project2(fine)
        middle = F.interpolate(
            self.project3(middle),
            size=output_size,
            mode="bilinear",
            align_corners=False,
        )
        deep = F.interpolate(
            self.project4(deep),
            size=output_size,
            mode="bilinear",
            align_corners=False,
        )

        fused = self.fusion(torch.cat([fine, middle, deep], dim=1))
        attended = self.self_attention(fused)
        return self.pool(attended).flatten(1)

    def forward(self, full_images, tile_images):
        full_features = self.encode_view(full_images)

        batch_size, tile_count, channels, height, width = tile_images.shape
        flat_tiles = tile_images.reshape(
            batch_size * tile_count, channels, height, width
        )
        tile_features = self.encode_view(flat_tiles).reshape(
            batch_size, tile_count, -1
        )

        tile_weights = torch.softmax(
            self.tile_score(tile_features).squeeze(-1),
            dim=1,
        )
        weighted_tiles = (
            tile_features * tile_weights.unsqueeze(-1)
        ).sum(dim=1)
        strongest_tile = tile_features.max(dim=1).values

        combined = torch.cat(
            [full_features, weighted_tiles, strongest_tile],
            dim=1,
        )
        return self.classifier(combined)


model = OkraResNet50Attention(num_classes=len(CLASS_NAMES), pretrained=True).to(DEVICE)

model.eval()
with torch.inference_mode():
    with torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=USE_AMP,
    ):
        sample_logits = model(
            full_batch[:1].to(DEVICE),
            tile_batch[:1].to(DEVICE),
        )

print("Device:", DEVICE)
print("Model:", model.__class__.__name__)
print("Output shape:", tuple(sample_logits.shape))
print("Output values finite:", bool(torch.isfinite(sample_logits).all()))
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 92.4MB/s]


Device: cuda
Model: OkraResNet50Attention
Output shape: (1, 5)
Output values finite: True
Trainable parameters: 26989382


In [10]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision.transforms import functional as TF
from PIL import Image, ImageOps, ImageEnhance, ImageFilter
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from tqdm.auto import tqdm
from pathlib import Path

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
BEST_CHECKPOINT_PATH = EXPERIMENT_DIR / "okra_best_model.pth"
LAST_CHECKPOINT_PATH = EXPERIMENT_DIR / "okra_last_checkpoint.pth"
HISTORY_PATH = EXPERIMENT_DIR / "okra_training_history.csv"

EPOCHS = 20
EARLY_STOPPING_PATIENCE = 8
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
BATCH_SIZE = 4
NUM_WORKERS = 0
USE_AMP = torch.cuda.is_available()

# Training-only augmentations; validation stays deterministic.
class OkraTrainingDataset(Dataset):
    def __init__(self, dataframe, training=False):
        self.df = dataframe.reset_index(drop=True)
        self.training = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        with Image.open(row["cached_image_path"]) as source:
            image = ImageOps.exif_transpose(source).convert("RGB")

        if self.training:
            if random.random() < 0.5:
                image = TF.hflip(image)
            if random.random() < 0.35:
                image = TF.rotate(
                    image,
                    random.uniform(-10, 10),
                    fill=(127, 127, 127),
                )
            if random.random() < 0.35:
                image = ImageEnhance.Color(image).enhance(random.uniform(0.85, 1.15))
                image = ImageEnhance.Brightness(image).enhance(random.uniform(0.9, 1.1))
                image = ImageEnhance.Contrast(image).enhance(random.uniform(0.9, 1.1))
            if random.random() < 0.05:
                image = image.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.1, 0.4)))

        full = TF.normalize(TF.to_tensor(image), IMAGENET_MEAN, IMAGENET_STD)

        tiles = []
        for left, top in tile_starts:
            tile = image.crop((left, top, left + TILE_SIZE, top + TILE_SIZE))
            tiles.append(
                TF.normalize(TF.to_tensor(tile), IMAGENET_MEAN, IMAGENET_STD)
            )

        label = CLASS_TO_IDX[row["label"]]
        return full, torch.stack(tiles), torch.tensor(label, dtype=torch.long)


# Balance sampling by class frequency, while keeping each epoch at 1,959 samples.
train_labels = train_df["label"].map(CLASS_TO_IDX).to_numpy()
class_counts = np.bincount(train_labels, minlength=len(CLASS_NAMES))
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[train_labels]

sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_df),
    replacement=True,
)

train_loader = DataLoader(
    OkraTrainingDataset(train_df, training=True),
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    OkraTrainingDataset(val_df, training=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-7,
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

start_epoch = 1
best_val_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0
history = []

# Resume automatically if a saved checkpoint exists.
if LAST_CHECKPOINT_PATH.exists():
    checkpoint = torch.load(
        LAST_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    if checkpoint.get("scaler_state_dict"):
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    start_epoch = int(checkpoint["epoch"]) + 1
    best_val_f1 = float(checkpoint.get("best_val_macro_f1", -1.0))
    best_epoch = int(checkpoint.get("best_epoch", 0))
    epochs_without_improvement = int(
        checkpoint.get("epochs_without_improvement", 0)
    )
    history = checkpoint.get("history", [])

    print(
        f"Resuming at epoch {start_epoch}; "
        f"best validation macro-F1={best_val_f1:.4f} at epoch {best_epoch}"
    )
else:
    print("Starting a fresh Okra training run.")

def run_epoch(loader, training):
    model.train(training)
    all_targets, all_predictions = [], []
    total_loss = 0.0
    total_count = 0

    progress = tqdm(
        loader,
        desc="Training" if training else "Validation",
        leave=True,
    )

    for full, tiles, targets in progress:
        full = full.to(DEVICE, non_blocking=True)
        tiles = tiles.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                logits = model(full, tiles)
                loss = criterion(logits, targets)

            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRAD_CLIP_NORM
                )
                scaler.step(optimizer)
                scaler.update()

        batch_size_now = targets.size(0)
        total_loss += loss.item() * batch_size_now
        total_count += batch_size_now

        predictions = logits.argmax(dim=1)
        all_targets.extend(targets.detach().cpu().tolist())
        all_predictions.extend(predictions.detach().cpu().tolist())

        progress.set_postfix(loss=f"{loss.item():.4f}")

    return {
        "loss": total_loss / max(total_count, 1),
        "accuracy": accuracy_score(all_targets, all_predictions),
        "balanced_accuracy": balanced_accuracy_score(
            all_targets, all_predictions
        ),
        "macro_f1": f1_score(
            all_targets,
            all_predictions,
            average="macro",
            labels=list(range(len(CLASS_NAMES))),
            zero_division=0,
        ),
    }

# Save an epoch-0 recovery point before the first training epoch.
if not LAST_CHECKPOINT_PATH.exists():
    torch.save(
        {
            "epoch": 0,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_val_macro_f1": best_val_f1,
            "best_epoch": best_epoch,
            "epochs_without_improvement": epochs_without_improvement,
            "history": history,
            "class_names": CLASS_NAMES,
        },
        LAST_CHECKPOINT_PATH,
    )

for epoch in range(start_epoch, EPOCHS + 1):
    current_lr = optimizer.param_groups[0]["lr"]

    train_metrics = run_epoch(train_loader, training=True)
    val_metrics = run_epoch(val_loader, training=False)
    scheduler.step(val_metrics["macro_f1"])

    row = {
        "epoch": epoch,
        "learning_rate": current_lr,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_balanced_accuracy": train_metrics["balanced_accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
    }
    history.append(row)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss={row['train_loss']:.4f}, "
        f"acc={row['train_accuracy']:.4f}, "
        f"macro-F1={row['train_macro_f1']:.4f} | "
        f"Val loss={row['val_loss']:.4f}, "
        f"acc={row['val_accuracy']:.4f}, "
        f"balanced acc={row['val_balanced_accuracy']:.4f}, "
        f"macro-F1={row['val_macro_f1']:.4f}"
    )

    improved = val_metrics["macro_f1"] > best_val_f1
    if improved:
        best_val_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "metrics": row,
                "class_names": CLASS_NAMES,
            },
            BEST_CHECKPOINT_PATH,
        )
        print("Saved new best checkpoint.")
    else:
        epochs_without_improvement += 1

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_val_macro_f1": best_val_f1,
            "best_epoch": best_epoch,
            "epochs_without_improvement": epochs_without_improvement,
            "history": history,
            "class_names": CLASS_NAMES,
        },
        LAST_CHECKPOINT_PATH,
    )
    print("Saved resumable checkpoint.")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping after {EARLY_STOPPING_PATIENCE} epochs without improvement.")
        break

print("\nTraining finished.")
print(f"Best validation macro-F1: {best_val_f1:.4f} at epoch {best_epoch}")
print("Best checkpoint:", BEST_CHECKPOINT_PATH)
print("Latest resumable checkpoint:", LAST_CHECKPOINT_PATH)
print("Training history:", HISTORY_PATH)

Resuming at epoch 11; best validation macro-F1=0.9204 at epoch 9


Training:   0%|          | 0/490 [00:00<?, ?it/s]

Validation:   0%|          | 0/105 [00:00<?, ?it/s]

Epoch 11/20 | Train loss=0.3404, acc=0.9581, macro-F1=0.9580 | Val loss=0.4562, acc=0.9190, balanced acc=0.9138, macro-F1=0.9089
Saved resumable checkpoint.


Training:   0%|          | 0/490 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/105 [00:00<?, ?it/s]

Epoch 19/20 | Train loss=0.2590, acc=0.9898, macro-F1=0.9897 | Val loss=0.4249, acc=0.9214, balanced acc=0.9131, macro-F1=0.9096
Saved resumable checkpoint.


Training:   0%|          | 0/490 [00:00<?, ?it/s]

Validation:   0%|          | 0/105 [00:00<?, ?it/s]

Epoch 20/20 | Train loss=0.2466, acc=0.9954, macro-F1=0.9953 | Val loss=0.4460, acc=0.9190, balanced acc=0.9100, macro-F1=0.9077
Saved resumable checkpoint.

Training finished.
Best validation macro-F1: 0.9257 at epoch 16
Best checkpoint: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_best_model.pth
Latest resumable checkpoint: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_last_checkpoint.pth
Training history: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_training_history.csv


In [11]:
import json
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
BEST_CHECKPOINT_PATH = EXPERIMENT_DIR / "okra_best_model.pth"

checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

if checkpoint.get("class_names") != CLASS_NAMES:
    raise ValueError(
        f"Checkpoint classes {checkpoint.get('class_names')} "
        f"do not match current classes {CLASS_NAMES}"
    )

model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.to(DEVICE).eval()

y_true, y_pred, confidences = [], [], []

with torch.inference_mode():
    for full, tiles, targets in tqdm(test_loader, desc="Okra internal test"):
        full = full.to(DEVICE, non_blocking=True)
        tiles = tiles.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            logits = model(full, tiles)

        probabilities = torch.softmax(logits.float(), dim=1)
        predictions = probabilities.argmax(dim=1)

        y_true.extend(targets.cpu().tolist())
        y_pred.extend(predictions.cpu().tolist())
        confidences.extend(probabilities.max(dim=1).values.cpu().tolist())

labels = list(range(len(CLASS_NAMES)))
report_dict = classification_report(
    y_true,
    y_pred,
    labels=labels,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
cm = confusion_matrix(y_true, y_pred, labels=labels)

metrics = {
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "test_images": len(y_true),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    "macro_f1": float(
        f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    ),
    "weighted_f1": float(
        f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
    ),
}

report_df = pd.DataFrame(report_dict).transpose()
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_df.index.name = "true_label"
cm_df.columns.name = "predicted_label"

# test_df is in the same sequential order as test_loader (shuffle=False).
prediction_df = test_df.reset_index(drop=True).copy()
prediction_df["true_label"] = [CLASS_NAMES[i] for i in y_true]
prediction_df["predicted_label"] = [CLASS_NAMES[i] for i in y_pred]
prediction_df["confidence"] = confidences
prediction_df["correct"] = prediction_df["true_label"] == prediction_df["predicted_label"]

with open(EXPERIMENT_DIR / "okra_internal_test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
report_df.to_csv(EXPERIMENT_DIR / "okra_internal_test_classification_report.csv")
cm_df.to_csv(EXPERIMENT_DIR / "okra_internal_test_confusion_matrix.csv")
prediction_df.to_csv(EXPERIMENT_DIR / "okra_internal_test_predictions.csv", index=False)

print(f"Best checkpoint epoch: {metrics['checkpoint_epoch']}")
print(f"Internal test images: {metrics['test_images']}")
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Balanced accuracy: {metrics['balanced_accuracy']:.4f}")
print(f"Macro-F1: {metrics['macro_f1']:.4f}")
print(f"Weighted-F1: {metrics['weighted_f1']:.4f}")

print("\nPer-class report:")
display(report_df.round(4))

print("\nConfusion matrix (true rows, predicted columns):")
display(cm_df)

print("\nSaved metrics, report, confusion matrix, and predictions to:")
print(EXPERIMENT_DIR)

Okra internal test:   0%|          | 0/105 [00:00<?, ?it/s]

Best checkpoint epoch: 16
Internal test images: 420
Accuracy: 0.9571
Balanced accuracy: 0.9561
Macro-F1: 0.9511
Weighted-F1: 0.9570

Per-class report:


,precision,recall,f1-score,support
Healthy_Leaf,0.9450,0.9904,0.9671,104.0000
Insects_Eaten_Leaf,0.9500,0.8837,0.9157,86.0000
Leaf_Curl,0.9273,0.9808,0.9533,52.0000
Leaf_Spot,0.9149,0.9556,0.9348,45.0000
Yellow_Vein_Mosaic_Virus,1.0000,0.9699,0.9847,133.0000
accuracy,0.9571,0.9571,0.9571,0.9571
macro avg,0.9474,0.9561,0.9511,420.0000
weighted avg,0.9580,0.9571,0.9570,420.0000



Confusion matrix (true rows, predicted columns):


predicted_label,Healthy_Leaf,Insects_Eaten_Leaf,Leaf_Curl,Leaf_Spot,Yellow_Vein_Mosaic_Virus
true_label,,,,,
Healthy_Leaf,103,0,1,0,0
Insects_Eaten_Leaf,5,76,1,4,0
Leaf_Curl,0,1,51,0,0
Leaf_Spot,0,1,1,43,0
Yellow_Vein_Mosaic_Virus,1,2,1,0,129



Saved metrics, report, confusion matrix, and predictions to:
/content/drive/MyDrive/final_year_project/okra/okra_model_experiment


In [12]:
from pathlib import Path
from collections import Counter

DRIVE_ROOT = Path("/content/drive/MyDrive/final_year_project")
SEARCH_ROOTS = [
    DRIVE_ROOT / "Tea" / "tea_ood_data",
    DRIVE_ROOT / "sugarcane" / "sugarcane_ood_data",
]

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

plantdoc_root = None
for search_root in SEARCH_ROOTS:
    if not search_root.exists():
        continue

    matches = [
        p for p in search_root.rglob("*")
        if p.is_dir() and p.name.lower() == "plantdoc-dataset-master"
    ]
    if matches:
        plantdoc_root = matches[0]
        break

if plantdoc_root is None:
    print("PlantDoc extracted folder not found in the expected Tea/Sugarcane Drive locations.")
    for root in SEARCH_ROOTS:
        print(f"{root}: exists={root.exists()}")
    raise FileNotFoundError(
        "Could not locate PlantDoc. Send me the output so I can point the next cell to its actual Drive path."
    )

print("PlantDoc folder:", plantdoc_root)

split_counts = {}
class_counts = Counter()
for split_dir in sorted(p for p in plantdoc_root.iterdir() if p.is_dir()):
    split_total = 0
    for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        count = sum(
            1 for p in class_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES
        )
        if count:
            class_counts[(split_dir.name, class_dir.name)] = count
            split_total += count
    split_counts[split_dir.name] = split_total

print("\nPlantDoc image count by split:")
for split_name, count in split_counts.items():
    print(f"  {split_name}: {count}")

print("\nPlantDoc images by split and class:")
for (split_name, class_name), count in sorted(class_counts.items()):
    print(f"  {split_name:10s} | {class_name:40s} | {count}")

print("\nTotal images:", sum(split_counts.values()))

PlantDoc folder: /content/drive/MyDrive/final_year_project/Tea/tea_ood_data/plantdoc_extract/PlantDoc-Dataset-master

PlantDoc image count by split:
  test: 236
  train: 2342

PlantDoc images by split and class:
  test       | Apple Scab Leaf                          | 10
  test       | Apple leaf                               | 9
  test       | Apple rust leaf                          | 10
  test       | Bell_pepper leaf                         | 8
  test       | Bell_pepper leaf spot                    | 9
  test       | Blueberry leaf                           | 11
  test       | Cherry leaf                              | 10
  test       | Corn Gray leaf spot                      | 4
  test       | Corn leaf blight                         | 12
  test       | Corn rust leaf                           | 10
  test       | Peach leaf                               | 9
  test       | Potato leaf early blight                 | 8
  test       | Potato leaf late blight                  | 8
  

In [13]:
from pathlib import Path
from collections import defaultdict
import hashlib

import pandas as pd
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
OKRA_MANIFEST = EXPERIMENT_DIR / "okra_final_2799_manifest_drive_paths.csv"
PLANTDOC_ROOT = Path(
    "/content/drive/MyDrive/final_year_project/Tea/tea_ood_data/"
    "plantdoc_extract/PlantDoc-Dataset-master"
)
PLANTDOC_AUDIT = EXPERIMENT_DIR / "okra_plantdoc_inventory_sha256.csv"
OVERLAP_REPORT = EXPERIMENT_DIR / "okra_plantdoc_exact_overlap_report.csv"
DUPLICATE_REPORT = EXPERIMENT_DIR / "okra_plantdoc_exact_duplicate_groups.csv"

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

if not OKRA_MANIFEST.exists():
    raise FileNotFoundError(OKRA_MANIFEST)
if not PLANTDOC_ROOT.exists():
    raise FileNotFoundError(PLANTDOC_ROOT)

okra_df = pd.read_csv(OKRA_MANIFEST)
if not {"image_path", "label"}.issubset(okra_df.columns):
    raise ValueError("Okra manifest must contain image_path and label.")

# Hash files only when the manifest does not already contain usable SHA-256 values.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

okra_hash_col = next(
    (c for c in okra_df.columns if c.lower() in {"sha256", "sha_256"}),
    None,
)
if okra_hash_col:
    okra_df["_sha256"] = okra_df[okra_hash_col].astype(str).str.lower()
else:
    okra_hashes = []
    for path in tqdm(okra_df["image_path"].astype(str), desc="Hashing retained Okra"):
        okra_hashes.append(sha256_file(path))
    okra_df["_sha256"] = okra_hashes

# Build PlantDoc inventory from its official split/class folders.
records = []
for split_dir in sorted(p for p in PLANTDOC_ROOT.iterdir() if p.is_dir()):
    for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        for image_path in class_dir.rglob("*"):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_SUFFIXES:
                records.append({
                    "image_path": str(image_path),
                    "plantdoc_class": class_dir.name,
                    "official_split": split_dir.name.lower(),
                })

plantdoc_df = pd.DataFrame(records)
if plantdoc_df.empty:
    raise RuntimeError("No PlantDoc images were found under the detected dataset folder.")

# Reuse a previously saved PlantDoc hash inventory if paths match.
existing_hash_files = [
    Path("/content/drive/MyDrive/final_year_project/Tea/tea_ood_data/plantdoc_inventory_manifest.csv"),
    Path("/content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/plantdoc_inventory_manifest.csv"),
]
reused = False
for hash_file in existing_hash_files:
    if not hash_file.exists():
        continue
    saved = pd.read_csv(hash_file)
    hash_col = next(
        (c for c in saved.columns if c.lower() in {"sha256", "sha_256"}),
        None,
    )
    if "image_path" in saved.columns and hash_col:
        lookup = dict(zip(saved["image_path"].astype(str),
                          saved[hash_col].astype(str).str.lower()))
        if plantdoc_df["image_path"].isin(lookup).all():
            plantdoc_df["_sha256"] = plantdoc_df["image_path"].map(lookup)
            reused = True
            print("Reused saved PlantDoc SHA-256 hashes from:", hash_file)
            break

if not reused:
    plantdoc_hashes = []
    for path in tqdm(plantdoc_df["image_path"], desc="Hashing PlantDoc images"):
        plantdoc_hashes.append(sha256_file(path))
    plantdoc_df["_sha256"] = plantdoc_hashes

# Exact Okra ↔ PlantDoc content overlap.
okra_hashes = set(okra_df["_sha256"])
overlap_hashes = okra_hashes.intersection(set(plantdoc_df["_sha256"]))
overlap_df = plantdoc_df[plantdoc_df["_sha256"].isin(overlap_hashes)].copy()

# Exact duplicate groups inside PlantDoc; report conflicting-label groups separately.
duplicate_rows = []
for sha, group in plantdoc_df.groupby("_sha256"):
    if len(group) > 1:
        duplicate_rows.append({
            "sha256": sha,
            "copies": len(group),
            "classes": " | ".join(sorted(group["plantdoc_class"].unique())),
            "splits": " | ".join(sorted(group["official_split"].unique())),
            "conflicting_classes": group["plantdoc_class"].nunique() > 1,
            "crosses_official_splits": group["official_split"].nunique() > 1,
            "image_paths": " | ".join(group["image_path"].astype(str)),
        })
duplicate_df = pd.DataFrame(duplicate_rows)

plantdoc_df = plantdoc_df.rename(columns={"_sha256": "sha256"})
plantdoc_df.to_csv(PLANTDOC_AUDIT, index=False)
overlap_df.rename(columns={"_sha256": "sha256"}).to_csv(OVERLAP_REPORT, index=False)
duplicate_df.to_csv(DUPLICATE_REPORT, index=False)

print("Okra images checked:", len(okra_df))
print("PlantDoc images checked:", len(plantdoc_df))
print("Exact Okra–PlantDoc matches:", len(overlap_df))
print("PlantDoc exact duplicate groups:", len(duplicate_df))
if not duplicate_df.empty:
    print("Groups with conflicting classes:",
          int(duplicate_df["conflicting_classes"].sum()))
    print("Groups crossing official splits:",
          int(duplicate_df["crosses_official_splits"].sum()))

print("\nPlantDoc inventory by split:")
print(plantdoc_df["official_split"].value_counts().sort_index().to_string())
print("\nAudit saved:", PLANTDOC_AUDIT)
print("Overlap report saved:", OVERLAP_REPORT)
print("PlantDoc duplicate groups saved:", DUPLICATE_REPORT)

if not overlap_df.empty:
    print("\nExact cross-dataset overlap examples:")
    display(overlap_df[["image_path", "plantdoc_class", "official_split", "sha256"]].head(20))
if not duplicate_df.empty:
    print("\nPlantDoc duplicate-group examples:")
    display(duplicate_df.head(20))

Hashing PlantDoc images:   0%|          | 0/2578 [00:00<?, ?it/s]

Okra images checked: 2799
PlantDoc images checked: 2578
Exact Okra–PlantDoc matches: 0
PlantDoc exact duplicate groups: 12
Groups with conflicting classes: 9
Groups crossing official splits: 11

PlantDoc inventory by split:
official_split
test      236
train    2342

Audit saved: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_plantdoc_inventory_sha256.csv
Overlap report saved: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_plantdoc_exact_overlap_report.csv
PlantDoc duplicate groups saved: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_plantdoc_exact_duplicate_groups.csv

PlantDoc duplicate-group examples:


,sha256,copies,classes,splits,conflicting_classes,crosses_official_splits,image_paths
0,07b9b0a3395d6643e0033f5327c188305479c20a67b9e0...,2,Blueberry leaf,test | train,False,True,/content/drive/MyDrive/final_year_project/Tea/...
1,3c52144e19e95cc3d4170407eaf729d0981b934615eaa1...,2,Tomato leaf yellow virus,test | train,False,True,/content/drive/MyDrive/final_year_project/Tea/...
2,64c0f7b5701e815769bad881c0834ffb3b58018233fc3a...,2,Potato leaf early blight | Potato leaf late bl...,test | train,True,True,/content/drive/MyDrive/final_year_project/Tea/...
3,6e777d1b6829140e66573f918c0f1681ca4cd2979859c3...,2,Corn Gray leaf spot | Corn leaf blight,test | train,True,True,/content/drive/MyDrive/final_year_project/Tea/...
4,9120578a76fd8539da220def6e0609ebb2435decc7a9c3...,2,Corn Gray leaf spot | Corn leaf blight,test | train,True,True,/content/drive/MyDrive/final_year_project/Tea/...
5,93c989d8551209bf34d4002ee9fae48ddbf6ab66fc3c27...,2,Tomato Septoria leaf spot,test | train,False,True,/content/drive/MyDrive/final_year_project/Tea/...
6,9dd93b4217495fbb725cfdded86d3dc5f98955c9616fff...,2,Potato leaf early blight | Potato leaf late bl...,test | train,True,True,/content/drive/MyDrive/final_year_project/Tea/...
7,a981ee59e5e1dcafd0d225f74369a05969200352c17743...,2,Potato leaf early blight | Potato leaf late bl...,test | train,True,True,/content/drive/MyDrive/final_year_project/Tea/...
8,c53ad8d40c38debfbc0b79a87d0f3649c20b933a65f5b7...,2,Potato leaf early blight | Potato leaf late bl...,test | train,True,True,/content/drive/MyDrive/final_year_project/Tea/...
9,e23a29c94b58aac28e92f57f95eb6f53e99c874323cb2b...,2,Tomato Septoria leaf spot | Tomato leaf bacter...,test | train,True,True,/content/drive/MyDrive/final_year_project/Tea/...


In [14]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
INVENTORY_PATH = EXPERIMENT_DIR / "okra_plantdoc_inventory_sha256.csv"

if not INVENTORY_PATH.exists():
    raise FileNotFoundError(INVENTORY_PATH)

df = pd.read_csv(INVENTORY_PATH)
required = {"image_path", "plantdoc_class", "official_split", "sha256"}
if not required.issubset(df.columns):
    raise ValueError(f"Inventory must contain: {sorted(required)}")

df["image_path"] = df["image_path"].astype(str)
df["official_split"] = df["official_split"].astype(str).str.lower()
df["sha256"] = df["sha256"].astype(str).str.lower()

# Handle exact duplicate content before any partitioning.
kept_indices = set(df.index)
set_aside_rows = []

for sha, group in df.groupby("sha256", sort=False):
    if len(group) <= 1:
        continue

    labels = group["plantdoc_class"].astype(str).unique().tolist()

    # Ambiguous exact copies with different class labels are all excluded.
    if len(labels) > 1:
        kept_indices.difference_update(group.index)
        for _, row in group.iterrows():
            set_aside_rows.append({
                "image_path": row["image_path"],
                "sha256": sha,
                "plantdoc_class": row["plantdoc_class"],
                "official_split": row["official_split"],
                "reason": "exact_duplicate_conflicting_class_labels",
            })
        continue

    # For same-class copies, retain one. If copies cross official splits,
    # prefer the official test image and set aside its train duplicate.
    ordered = group.assign(
        _split_priority=group["official_split"].map({"test": 0, "train": 1}).fillna(2)
    ).sort_values(["_split_priority", "image_path"])
    keep_index = ordered.index[0]

    for idx, row in group.iterrows():
        if idx != keep_index:
            kept_indices.discard(idx)
            set_aside_rows.append({
                "image_path": row["image_path"],
                "sha256": sha,
                "plantdoc_class": row["plantdoc_class"],
                "official_split": row["official_split"],
                "reason": f"redundant_exact_copy_kept:{df.loc[keep_index, 'image_path']}",
            })

clean = df.loc[sorted(kept_indices)].copy()
train_pool = clean[clean["official_split"] == "train"].copy()
ood_test = clean[clean["official_split"] == "test"].copy()

if train_pool.empty or ood_test.empty:
    raise RuntimeError("PlantDoc train or official test split is empty after duplicate cleanup.")

# Deterministic within-class split of official PlantDoc train:
# roughly 70% OOD train, 15% calibration, 15% validation.
rng = np.random.default_rng(42)
train_parts, calibration_parts, validation_parts = [], [], []

for class_name, group in train_pool.groupby("plantdoc_class", sort=True):
    indices = group.index.to_numpy().copy()
    rng.shuffle(indices)
    n = len(indices)

    # Very small classes remain training-only so they don't disappear from OE.
    if n < 3:
        n_cal = 0
        n_val = 0
    else:
        n_cal = max(1, int(round(n * 0.15)))
        n_val = max(1, int(round(n * 0.15)))
        while n_cal + n_val >= n:
            if n_cal >= n_val and n_cal > 1:
                n_cal -= 1
            elif n_val > 1:
                n_val -= 1
            else:
                n_cal = n_val = 0
                break

    n_train = n - n_cal - n_val
    train_parts.append(group.loc[indices[:n_train]])
    if n_cal:
        calibration_parts.append(group.loc[indices[n_train:n_train + n_cal]])
    if n_val:
        validation_parts.append(group.loc[indices[n_train + n_cal:]])

ood_train = pd.concat(train_parts).copy()
ood_calibration = (
    pd.concat(calibration_parts).copy()
    if calibration_parts else train_pool.iloc[0:0].copy()
)
ood_validation = (
    pd.concat(validation_parts).copy()
    if validation_parts else train_pool.iloc[0:0].copy()
)

# Add consistent metadata and store each manifest.
split_frames = {
    "train": ood_train,
    "calibration": ood_calibration,
    "validation": ood_validation,
    "test": ood_test,
}
for split_name, part in split_frames.items():
    part = part.copy()
    part["ood_source"] = "plantdoc_other_crop"
    part["ood_split"] = split_name
    part.to_csv(
        EXPERIMENT_DIR / f"okra_plantdoc_ood_{split_name}_manifest.csv",
        index=False,
    )
    split_frames[split_name] = part

set_aside_path = EXPERIMENT_DIR / "okra_plantdoc_exact_duplicates_set_aside.csv"
pd.DataFrame(set_aside_rows).to_csv(set_aside_path, index=False)

# Verify disjoint files and no SHA-256 content leaks between partitions.
for left_name, right_name in [
    ("train", "calibration"),
    ("train", "validation"),
    ("train", "test"),
    ("calibration", "validation"),
    ("calibration", "test"),
    ("validation", "test"),
]:
    left = split_frames[left_name]
    right = split_frames[right_name]
    assert set(left["sha256"]).isdisjoint(set(right["sha256"])), (
        f"Exact content overlap between {left_name} and {right_name}"
    )

print("PlantDoc images before duplicate cleanup:", len(df))
print("Exact duplicate copies/groups set aside:", len(set_aside_rows))
print("Clean PlantDoc images:", len(clean))
print("\nOOD split totals:")
for split_name, part in split_frames.items():
    print(f"  {split_name}: {len(part)}")

print("\nPer-class counts by OOD split:")
counts = pd.concat(
    [part.assign(split=name) for name, part in split_frames.items()],
    ignore_index=True,
)
display(pd.crosstab(counts["split"], counts["plantdoc_class"]))

small_classes = (
    ood_train["plantdoc_class"].value_counts()
    .loc[lambda values: values < 3]
    .to_dict()
)
print("\nSmall classes kept in OOD training only:", small_classes)
print("\nSaved manifests in:", EXPERIMENT_DIR)
print("Duplicate set-aside log:", set_aside_path)

PlantDoc images before duplicate cleanup: 2578
Exact duplicate copies/groups set aside: 21
Clean PlantDoc images: 2557

OOD split totals:
  train: 1631
  calibration: 349
  validation: 349
  test: 228

Per-class counts by OOD split:


plantdoc_class,Apple Scab Leaf,Apple leaf,Apple rust leaf,Bell_pepper leaf,Bell_pepper leaf spot,Blueberry leaf,Cherry leaf,Corn Gray leaf spot,Corn leaf blight,Corn rust leaf,...,Tomato Septoria leaf spot,Tomato leaf,Tomato leaf bacterial spot,Tomato leaf late blight,Tomato leaf mosaic virus,Tomato leaf yellow virus,Tomato mold leaf,Tomato two spotted spider mites leaf,grape leaf,grape leaf black rot
split,,,,,,,,,,,,,,,,,,,,,
calibration,12,12,12,8,9,16,7,9,27,16,...,21,8,15,15,7,10,13,0,9,8
test,10,9,10,8,9,11,10,3,10,10,...,11,8,8,10,10,6,6,0,12,8
train,59,58,55,37,44,73,33,44,125,74,...,96,39,71,71,30,49,59,2,39,40
validation,12,12,12,8,9,16,7,9,27,16,...,21,8,15,15,7,10,13,0,9,8



Small classes kept in OOD training only: {'Tomato two spotted spider mites leaf': 2}

Saved manifests in: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment
Duplicate set-aside log: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_plantdoc_exact_duplicates_set_aside.csv


In [15]:
from pathlib import Path
import shutil
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from PIL import Image, ImageOps

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
OOD_DIR = EXPERIMENT_DIR / "ood_data"
OOD_DIR.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT_PATH = EXPERIMENT_DIR / "okra_best_model.pth"
BASELINE_CHECKPOINT_PATH = OOD_DIR / "okra_baseline_before_ood.pth"

if not BEST_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Best supervised checkpoint not found: {BEST_CHECKPOINT_PATH}")

baseline_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(baseline_checkpoint["model_state_dict"], strict=True)
model.to(DEVICE)

# Save the supervised model unchanged as the OE starting point.
torch.save(
    {
        "epoch": baseline_checkpoint["epoch"],
        "model_state_dict": baseline_checkpoint["model_state_dict"],
        "metrics": baseline_checkpoint.get("metrics", {}),
        "class_names": CLASS_NAMES,
        "description": "Untouched supervised Okra checkpoint before outlier exposure.",
    },
    BASELINE_CHECKPOINT_PATH,
)

class OkraOODDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image_path = str(row["image_path"])

        with Image.open(image_path) as source:
            image = ImageOps.exif_transpose(source).convert("RGB")
            image = ImageOps.pad(
                image,
                (IMAGE_SIZE, IMAGE_SIZE),
                method=Image.Resampling.LANCZOS,
                color=(127, 127, 127),
            )

        full = TF.normalize(TF.to_tensor(image), IMAGENET_MEAN, IMAGENET_STD)

        tiles = []
        for left, top in tile_starts:
            tile = image.crop((left, top, left + TILE_SIZE, top + TILE_SIZE))
            tiles.append(
                TF.normalize(TF.to_tensor(tile), IMAGENET_MEAN, IMAGENET_STD)
            )

        return full, torch.stack(tiles)

def load_ood_split(split_name):
    path = EXPERIMENT_DIR / f"okra_plantdoc_ood_{split_name}_manifest.csv"
    if not path.exists():
        raise FileNotFoundError(path)

    frame = pd.read_csv(path)
    if "image_path" not in frame.columns:
        raise ValueError(f"{path.name} has no image_path column.")

    missing = [p for p in frame["image_path"].astype(str) if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError(
            f"{len(missing)} PlantDoc files are missing. Examples: {missing[:5]}"
        )
    return frame

ood_train_df = load_ood_split("train")
ood_calibration_df = load_ood_split("calibration")
ood_validation_df = load_ood_split("validation")

ood_train_loader = DataLoader(
    OkraOODDataset(ood_train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
ood_validation_loader = DataLoader(
    OkraOODDataset(ood_validation_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

ood_full_batch, ood_tile_batch = next(iter(ood_train_loader))

print("Untouched supervised baseline:", BASELINE_CHECKPOINT_PATH)
print("Baseline checkpoint epoch:", baseline_checkpoint["epoch"])
print("Supported train images:", len(train_df))
print("Supported validation images:", len(val_df))
print("PlantDoc OOD train images:", len(ood_train_df))
print("PlantDoc OOD calibration images:", len(ood_calibration_df))
print("PlantDoc OOD validation images:", len(ood_validation_df))
print("PlantDoc OOD test images: 228 (manifest remains unused)")
print("OOD full batch:", tuple(ood_full_batch.shape))
print("OOD tile batch:", tuple(ood_tile_batch.shape))

Untouched supervised baseline: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/ood_data/okra_baseline_before_ood.pth
Baseline checkpoint epoch: 16
Supported train images: 1959
Supported validation images: 420
PlantDoc OOD train images: 1631
PlantDoc OOD calibration images: 349
PlantDoc OOD validation images: 349
PlantDoc OOD test images: 228 (manifest remains unused)
OOD full batch: (4, 3, 448, 448)
OOD tile batch: (4, 4, 3, 288, 288)


In [16]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

OOD_DIR = EXPERIMENT_DIR / "ood_data"
BASELINE_CHECKPOINT_PATH = OOD_DIR / "okra_baseline_before_ood.pth"
OE_BEST_PATH = OOD_DIR / "okra_oe_best.pth"
OE_LAST_PATH = OOD_DIR / "okra_oe_last.pth"
OE_HISTORY_PATH = OOD_DIR / "okra_oe_training_history.csv"

OE_EPOCHS = 3
OE_LEARNING_RATE = 1e-6
OE_WEIGHT = 0.5
MAX_SUPPORTED_MACRO_F1_DROP = 0.02
GRAD_CLIP_NORM = 1.0

baseline = torch.load(
    BASELINE_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(baseline["model_state_dict"], strict=True)
model.to(DEVICE)

oe_optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=OE_LEARNING_RATE,
    weight_decay=1e-4,
)
oe_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def collect_supported_outputs(loader, description):
    model.eval()
    y_true, y_pred, confidence = [], [], []
    with torch.inference_mode():
        for full, tiles, targets in tqdm(loader, desc=description):
            full = full.to(DEVICE, non_blocking=True)
            tiles = tiles.to(DEVICE, non_blocking=True)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                logits = model(full, tiles)
            probabilities = torch.softmax(logits.float(), dim=1)
            y_true.extend(targets.tolist())
            y_pred.extend(probabilities.argmax(dim=1).cpu().tolist())
            confidence.extend(probabilities.max(dim=1).values.cpu().tolist())

    metrics_out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(
            f1_score(
                y_true, y_pred, average="macro",
                labels=list(range(len(CLASS_NAMES))), zero_division=0
            )
        ),
    }
    return metrics_out, np.asarray(confidence, dtype=np.float32)

def collect_ood_confidence(loader, description):
    model.eval()
    confidence = []
    with torch.inference_mode():
        for full, tiles in tqdm(loader, desc=description):
            full = full.to(DEVICE, non_blocking=True)
            tiles = tiles.to(DEVICE, non_blocking=True)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                logits = model(full, tiles)
            confidence.extend(
                torch.softmax(logits.float(), dim=1).max(dim=1).values.cpu().tolist()
            )
    return np.asarray(confidence, dtype=np.float32)

# Baseline fallback and validation reference.
baseline_val_metrics, baseline_supported_conf = collect_supported_outputs(
    val_loader, "Baseline Okra validation"
)
baseline_ood_conf = collect_ood_confidence(
    ood_validation_loader, "Baseline PlantDoc validation"
)
baseline_threshold = float(np.quantile(baseline_supported_conf, 0.05))
baseline_ood_rejection = float(np.mean(baseline_ood_conf < baseline_threshold))

best_ood_rejection = baseline_ood_rejection
best_epoch = 0
best_supported_macro_f1 = baseline_val_metrics["macro_f1"]
history = []

torch.save(
    {
        "epoch": 0,
        "model_state_dict": baseline["model_state_dict"],
        "metrics": {
            "supported_validation": baseline_val_metrics,
            "confidence_threshold_at_95pct_supported_acceptance": baseline_threshold,
            "plantdoc_validation_rejection": baseline_ood_rejection,
        },
        "class_names": CLASS_NAMES,
        "description": "Baseline fallback before OE improvement.",
    },
    OE_BEST_PATH,
)

start_epoch = 1
if OE_LAST_PATH.exists():
    saved = torch.load(OE_LAST_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(saved["model_state_dict"], strict=True)
    oe_optimizer.load_state_dict(saved["optimizer_state_dict"])
    if saved.get("scaler_state_dict"):
        oe_scaler.load_state_dict(saved["scaler_state_dict"])
    start_epoch = int(saved["epoch"]) + 1
    best_ood_rejection = float(saved["best_ood_rejection"])
    best_epoch = int(saved["best_epoch"])
    best_supported_macro_f1 = float(saved["best_supported_macro_f1"])
    history = saved.get("history", [])
    print(f"Resuming OE at epoch {start_epoch}; best OE epoch={best_epoch}")
else:
    print(
        "Starting from supervised baseline | "
        f"validation macro-F1={baseline_val_metrics['macro_f1']:.4f} | "
        f"PlantDoc validation rejection={baseline_ood_rejection:.1%}"
    )

steps_per_epoch = max(len(train_loader), len(ood_train_loader))
uniform_target = None

for epoch in range(start_epoch, OE_EPOCHS + 1):
    model.train()
    supported_iter = iter(train_loader)
    ood_iter = iter(ood_train_loader)
    total_supported_loss = 0.0
    total_ood_loss = 0.0
    seen = 0

    progress = tqdm(range(steps_per_epoch), desc=f"Okra OE epoch {epoch}/{OE_EPOCHS}")
    for _ in progress:
        try:
            full, tiles, targets = next(supported_iter)
        except StopIteration:
            supported_iter = iter(train_loader)
            full, tiles, targets = next(supported_iter)

        try:
            ood_full, ood_tiles = next(ood_iter)
        except StopIteration:
            ood_iter = iter(ood_train_loader)
            ood_full, ood_tiles = next(ood_iter)

        full = full.to(DEVICE, non_blocking=True)
        tiles = tiles.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        ood_full = ood_full.to(DEVICE, non_blocking=True)
        ood_tiles = ood_tiles.to(DEVICE, non_blocking=True)

        oe_optimizer.zero_grad(set_to_none=True)
        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            supported_logits = model(full, tiles)
            ood_logits = model(ood_full, ood_tiles)

            supported_loss = F.cross_entropy(supported_logits, targets)
            log_probs = F.log_softmax(ood_logits.float(), dim=1)
            uniform = torch.full_like(log_probs, 1.0 / len(CLASS_NAMES))
            ood_loss = F.kl_div(log_probs, uniform, reduction="batchmean")
            loss = supported_loss + OE_WEIGHT * ood_loss

        oe_scaler.scale(loss).backward()
        oe_scaler.unscale_(oe_optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        oe_scaler.step(oe_optimizer)
        oe_scaler.update()

        batch_n = targets.size(0)
        total_supported_loss += supported_loss.item() * batch_n
        total_ood_loss += ood_loss.item() * batch_n
        seen += batch_n
        progress.set_postfix(
            supported_loss=f"{supported_loss.item():.3f}",
            ood_loss=f"{ood_loss.item():.3f}",
        )

    val_metrics, supported_conf = collect_supported_outputs(
        val_loader, "Okra validation"
    )
    ood_conf = collect_ood_confidence(
        ood_validation_loader, "PlantDoc OOD validation"
    )

    # Diagnostic operating point only; final threshold is calibrated separately.
    threshold = float(np.quantile(supported_conf, 0.05))
    supported_acceptance = float(np.mean(supported_conf >= threshold))
    ood_rejection = float(np.mean(ood_conf < threshold))

    row = {
        "epoch": epoch,
        "supported_train_loss": total_supported_loss / max(seen, 1),
        "ood_train_kl_loss": total_ood_loss / max(seen, 1),
        "supported_val_accuracy": val_metrics["accuracy"],
        "supported_val_balanced_accuracy": val_metrics["balanced_accuracy"],
        "supported_val_macro_f1": val_metrics["macro_f1"],
        "diagnostic_confidence_threshold": threshold,
        "supported_val_acceptance": supported_acceptance,
        "plantdoc_val_rejection": ood_rejection,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(OE_HISTORY_PATH, index=False)

    print(
        f"Epoch {epoch} | supported train loss={row['supported_train_loss']:.4f} | "
        f"OOD KL={row['ood_train_kl_loss']:.4f} | "
        f"supported val macro-F1={val_metrics['macro_f1']:.4f} | "
        f"PlantDoc val rejection={ood_rejection:.1%} | "
        f"threshold={threshold:.4f}"
    )

    eligible = (
        val_metrics["macro_f1"]
        >= baseline_val_metrics["macro_f1"] - MAX_SUPPORTED_MACRO_F1_DROP
    )
    improved = eligible and ood_rejection > best_ood_rejection

    if improved:
        best_ood_rejection = ood_rejection
        best_epoch = epoch
        best_supported_macro_f1 = val_metrics["macro_f1"]
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "metrics": row,
                "class_names": CLASS_NAMES,
                "description": "Best OE checkpoint selected on validation data.",
            },
            OE_BEST_PATH,
        )
        print("Saved improved OE checkpoint.")
    elif not eligible:
        print("Not eligible as best: supported validation macro-F1 dropped too far.")

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": oe_optimizer.state_dict(),
            "scaler_state_dict": oe_scaler.state_dict(),
            "best_ood_rejection": best_ood_rejection,
            "best_epoch": best_epoch,
            "best_supported_macro_f1": best_supported_macro_f1,
            "history": history,
            "class_names": CLASS_NAMES,
        },
        OE_LAST_PATH,
    )
    print("Saved resumable OE checkpoint.")

print("\nOE fine-tuning finished.")
print("Best selected epoch:", best_epoch)
print(f"Best validation PlantDoc rejection: {best_ood_rejection:.1%}")
print(f"Supported validation macro-F1: {best_supported_macro_f1:.4f}")
print("Best OE checkpoint:", OE_BEST_PATH)
print("Latest resumable checkpoint:", OE_LAST_PATH)
print("Training history:", OE_HISTORY_PATH)

Baseline Okra validation:   0%|          | 0/105 [00:00<?, ?it/s]

Baseline PlantDoc validation:   0%|          | 0/88 [00:00<?, ?it/s]

Starting from supervised baseline | validation macro-F1=0.9257 | PlantDoc validation rejection=55.3%


Okra OE epoch 1/3:   0%|          | 0/490 [00:00<?, ?it/s]

Okra validation:   0%|          | 0/105 [00:00<?, ?it/s]

...

Epoch 2 | supported train loss=0.1582 | OOD KL=0.4177 | supported val macro-F1=0.9141 | PlantDoc val rejection=61.9% | threshold=0.5805
Saved resumable OE checkpoint.


Okra OE epoch 3/3:   0%|          | 0/490 [00:00<?, ?it/s]

Okra validation:   0%|          | 0/105 [00:00<?, ?it/s]

PlantDoc OOD validation:   0%|          | 0/88 [00:00<?, ?it/s]

Epoch 3 | supported train loss=0.1748 | OOD KL=0.2865 | supported val macro-F1=0.9097 | PlantDoc val rejection=58.2% | threshold=0.5703
Saved resumable OE checkpoint.

OE fine-tuning finished.
Best selected epoch: 1
Best validation PlantDoc rejection: 61.9%
Supported validation macro-F1: 0.9076
Best OE checkpoint: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/ood_data/okra_oe_best.pth
Latest resumable checkpoint: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/ood_data/okra_oe_last.pth
Training history: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/ood_data/okra_oe_training_history.csv


In [17]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

OOD_DIR = EXPERIMENT_DIR / "ood_data"
OE_BEST_PATH = OOD_DIR / "okra_oe_best.pth"
FEATURES_PATH = OOD_DIR / "okra_oe_knn_calibration_embeddings.pt"
SCORES_PATH = OOD_DIR / "okra_oe_knn_calibration_scores.csv"

checkpoint = torch.load(OE_BEST_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.to(DEVICE).eval()

# Deterministic supported loaders; training augmentation is disabled for feature extraction.
supported_train_features_loader = DataLoader(
    OkraImageDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
supported_val_features_loader = DataLoader(
    OkraImageDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
ood_calibration_features_loader = DataLoader(
    OkraOODDataset(ood_calibration_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
ood_validation_features_loader = DataLoader(
    OkraOODDataset(ood_validation_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

def represent_and_score(full, tiles):
    full_features = model.encode_view(full)

    batch_size, tile_count, channels, height, width = tiles.shape
    flat_tiles = tiles.reshape(batch_size * tile_count, channels, height, width)
    tile_features = model.encode_view(flat_tiles).reshape(batch_size, tile_count, -1)

    tile_weights = torch.softmax(
        model.tile_score(tile_features).squeeze(-1), dim=1
    )
    weighted_tiles = (tile_features * tile_weights.unsqueeze(-1)).sum(dim=1)
    strongest_tile = tile_features.max(dim=1).values

    logits = model.classifier(
        torch.cat([full_features, weighted_tiles, strongest_tile], dim=1)
    )
    confidence = torch.softmax(logits.float(), dim=1).max(dim=1).values
    embedding = F.normalize(full_features.float(), dim=1)
    return embedding, confidence

def extract_supported(loader, dataframe, description):
    embeddings, confidences, labels = [], [], []
    with torch.inference_mode():
        for full, tiles, target in tqdm(loader, desc=description):
            full = full.to(DEVICE, non_blocking=True)
            tiles = tiles.to(DEVICE, non_blocking=True)
            embedding, confidence = represent_and_score(full, tiles)
            embeddings.append(embedding.cpu())
            confidences.extend(confidence.cpu().tolist())
            labels.extend(target.tolist())

    paths = dataframe["image_path"].astype(str).tolist()
    if len(paths) != len(labels):
        raise RuntimeError(f"{description}: path and output counts do not match.")
    return (
        torch.cat(embeddings),
        np.asarray(confidences, dtype=np.float32),
        np.asarray(labels, dtype=np.int64),
        paths,
    )

def extract_ood(loader, dataframe, description):
    embeddings, confidences = [], []
    with torch.inference_mode():
        for full, tiles in tqdm(loader, desc=description):
            full = full.to(DEVICE, non_blocking=True)
            tiles = tiles.to(DEVICE, non_blocking=True)
            embedding, confidence = represent_and_score(full, tiles)
            embeddings.append(embedding.cpu())
            confidences.extend(confidence.cpu().tolist())

    paths = dataframe["image_path"].astype(str).tolist()
    if len(paths) != len(confidences):
        raise RuntimeError(f"{description}: path and output counts do not match.")
    return (
        torch.cat(embeddings),
        np.asarray(confidences, dtype=np.float32),
        paths,
    )

ref_embeddings, ref_conf, ref_labels, ref_paths = extract_supported(
    supported_train_features_loader,
    train_df,
    "Okra training reference features",
)
val_embeddings, val_conf, val_labels, val_paths = extract_supported(
    supported_val_features_loader,
    val_df,
    "Okra validation calibration features",
)
cal_embeddings, cal_conf, cal_paths = extract_ood(
    ood_calibration_features_loader,
    ood_calibration_df,
    "PlantDoc calibration features",
)
ood_val_embeddings, ood_val_conf, ood_val_paths = extract_ood(
    ood_validation_features_loader,
    ood_validation_df,
    "PlantDoc validation features",
)

features = {
    "reference_embeddings": ref_embeddings,
    "reference_labels": torch.as_tensor(ref_labels, dtype=torch.long),
    "reference_paths": ref_paths,
    "supported_validation_embeddings": val_embeddings,
    "supported_validation_labels": torch.as_tensor(val_labels, dtype=torch.long),
    "supported_validation_paths": val_paths,
    "plantdoc_calibration_embeddings": cal_embeddings,
    "plantdoc_calibration_paths": cal_paths,
    "plantdoc_validation_embeddings": ood_val_embeddings,
    "plantdoc_validation_paths": ood_val_paths,
    "checkpoint": str(OE_BEST_PATH),
    "k": 10,
}
torch.save(features, FEATURES_PATH)

score_rows = []
for i, path in enumerate(val_paths):
    score_rows.append({
        "image_path": path,
        "split": "supported_validation",
        "label": CLASS_NAMES[int(val_labels[i])],
        "confidence": float(val_conf[i]),
    })
for i, path in enumerate(cal_paths):
    score_rows.append({
        "image_path": path,
        "split": "plantdoc_calibration",
        "label": None,
        "confidence": float(cal_conf[i]),
    })
for i, path in enumerate(ood_val_paths):
    score_rows.append({
        "image_path": path,
        "split": "plantdoc_validation",
        "label": None,
        "confidence": float(ood_val_conf[i]),
    })

pd.DataFrame(score_rows).to_csv(SCORES_PATH, index=False)

print("OE checkpoint epoch:", checkpoint["epoch"])
print("Reference embeddings:", tuple(ref_embeddings.shape))
print("Supported validation embeddings:", tuple(val_embeddings.shape))
print("PlantDoc calibration embeddings:", tuple(cal_embeddings.shape))
print("PlantDoc validation embeddings:", tuple(ood_val_embeddings.shape))
print(f"\nSaved feature bank: {FEATURES_PATH}")
print(f"Saved confidence scores: {SCORES_PATH}")

Okra training reference features:   0%|          | 0/490 [00:00<?, ?it/s]

Okra validation calibration features:   0%|          | 0/105 [00:00<?, ?it/s]

PlantDoc calibration features:   0%|          | 0/88 [00:00<?, ?it/s]

PlantDoc validation features:   0%|          | 0/88 [00:00<?, ?it/s]

OE checkpoint epoch: 1
Reference embeddings: (1959, 256)
Supported validation embeddings: (420, 256)
PlantDoc calibration embeddings: (349, 256)
PlantDoc validation embeddings: (349, 256)

Saved feature bank: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/ood_data/okra_oe_knn_calibration_embeddings.pt
Saved confidence scores: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/ood_data/okra_oe_knn_calibration_scores.csv


In [18]:
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path

OOD_DIR = EXPERIMENT_DIR / "ood_data"
FEATURES_PATH = OOD_DIR / "okra_oe_knn_calibration_embeddings.pt"
SCORES_PATH = OOD_DIR / "okra_oe_knn_calibration_scores.csv"

K = 10
TARGET_SUPPORTED_ACCEPTANCE = 0.95

features = torch.load(FEATURES_PATH, map_location="cpu", weights_only=False)

reference = torch.nn.functional.normalize(
    features["reference_embeddings"].float(), dim=1
)
supported_embeddings = torch.nn.functional.normalize(
    features["supported_validation_embeddings"].float(), dim=1
)
plantdoc_cal_embeddings = torch.nn.functional.normalize(
    features["plantdoc_calibration_embeddings"].float(), dim=1
)
plantdoc_val_embeddings = torch.nn.functional.normalize(
    features["plantdoc_validation_embeddings"].float(), dim=1
)

supported_labels = features["supported_validation_labels"].numpy().astype(int)
supported_paths = features["supported_validation_paths"]
plantdoc_cal_paths = features["plantdoc_calibration_paths"]
plantdoc_val_paths = features["plantdoc_validation_paths"]

score_rows = pd.read_csv(SCORES_PATH)
supported_conf = score_rows.loc[
    score_rows["split"] == "supported_validation", "confidence"
].to_numpy(dtype=np.float32)
plantdoc_cal_conf = score_rows.loc[
    score_rows["split"] == "plantdoc_calibration", "confidence"
].to_numpy(dtype=np.float32)
plantdoc_val_conf = score_rows.loc[
    score_rows["split"] == "plantdoc_validation", "confidence"
].to_numpy(dtype=np.float32)

if len(supported_conf) != len(supported_embeddings):
    raise ValueError("Supported confidence and embedding counts do not match.")
if len(plantdoc_cal_conf) != len(plantdoc_cal_embeddings):
    raise ValueError("PlantDoc calibration confidence and embedding counts do not match.")
if len(plantdoc_val_conf) != len(plantdoc_val_embeddings):
    raise ValueError("PlantDoc validation confidence and embedding counts do not match.")

def knn_similarity(query_embeddings, reference_embeddings, k=10, query_batch=256):
    result = []
    for start in range(0, len(query_embeddings), query_batch):
        query = query_embeddings[start:start + query_batch]
        similarities = query @ reference_embeddings.T
        values = similarities.topk(min(k, reference_embeddings.shape[0]), dim=1).values
        result.extend(values.mean(dim=1).numpy().tolist())
    return np.asarray(result, dtype=np.float32)

supported_knn = knn_similarity(supported_embeddings, reference, K)
plantdoc_cal_knn = knn_similarity(plantdoc_cal_embeddings, reference, K)
plantdoc_val_knn = knn_similarity(plantdoc_val_embeddings, reference, K)

# Threshold grid spans the low-score tail that can preserve >=95% supported acceptance.
quantiles = np.linspace(0.0, 0.05, 101)
conf_thresholds = np.unique(np.quantile(supported_conf, quantiles))
knn_thresholds = np.unique(np.quantile(supported_knn, quantiles))

# Sweep combined thresholds. An image is accepted only when both tests pass.
sweep_rows = []
for conf_threshold in conf_thresholds:
    for knn_threshold in knn_thresholds:
        supported_accept = (
            (supported_conf >= conf_threshold)
            & (supported_knn >= knn_threshold)
        )
        supported_rate = float(supported_accept.mean())
        if supported_rate < TARGET_SUPPORTED_ACCEPTANCE:
            continue

        plantdoc_reject = float(
            (
                (plantdoc_cal_conf < conf_threshold)
                | (plantdoc_cal_knn < knn_threshold)
            ).mean()
        )

        per_class_rates = {}
        for class_index, class_name in enumerate(CLASS_NAMES):
            mask = supported_labels == class_index
            per_class_rates[class_name] = (
                float(supported_accept[mask].mean()) if mask.any() else float("nan")
            )

        sweep_rows.append({
            "confidence_threshold": float(conf_threshold),
            "knn_threshold": float(knn_threshold),
            "supported_acceptance": supported_rate,
            "plantdoc_calibration_rejection": plantdoc_reject,
            "minimum_class_acceptance": float(np.nanmin(list(per_class_rates.values()))),
            **{f"accept_{name}": rate for name, rate in per_class_rates.items()},
        })

if not sweep_rows:
    raise RuntimeError(
        "No combined threshold reached the supported acceptance target. "
        "Inspect confidence and kNN score distributions before lowering the target."
    )

sweep = pd.DataFrame(sweep_rows).sort_values(
    [
        "plantdoc_calibration_rejection",
        "supported_acceptance",
        "minimum_class_acceptance",
    ],
    ascending=[False, False, False],
).reset_index(drop=True)

selected = sweep.iloc[0]
conf_threshold = float(selected["confidence_threshold"])
knn_threshold = float(selected["knn_threshold"])

def evaluate_policy(confidence, knn_score, c_threshold, k_threshold):
    accepted = (confidence >= c_threshold) & (knn_score >= k_threshold)
    return accepted

supported_accepted = evaluate_policy(
    supported_conf, supported_knn, conf_threshold, knn_threshold
)
plantdoc_cal_rejected = ~evaluate_policy(
    plantdoc_cal_conf, plantdoc_cal_knn, conf_threshold, knn_threshold
)
plantdoc_val_rejected = ~evaluate_policy(
    plantdoc_val_conf, plantdoc_val_knn, conf_threshold, knn_threshold
)

per_class_acceptance = {}
for class_index, class_name in enumerate(CLASS_NAMES):
    mask = supported_labels == class_index
    per_class_acceptance[class_name] = float(supported_accepted[mask].mean())

config = {
    "crop": "okra",
    "model_type": "outlier_exposure_plus_knn",
    "checkpoint": str(OOD_DIR / "okra_oe_best.pth"),
    "k": K,
    "policy": "accept if confidence >= threshold AND mean top-k cosine similarity >= threshold",
    "confidence_threshold": conf_threshold,
    "knn_threshold": knn_threshold,
    "target_supported_acceptance": TARGET_SUPPORTED_ACCEPTANCE,
    "calibration_metrics": {
        "supported_validation_acceptance": float(supported_accepted.mean()),
        "supported_acceptance_by_class": per_class_acceptance,
        "plantdoc_calibration_rejection": float(plantdoc_cal_rejected.mean()),
    },
    "plantdoc_validation_diagnostic_rejection": float(plantdoc_val_rejected.mean()),
    "classes": CLASS_NAMES,
}

sweep_path = OOD_DIR / "okra_oe_knn_threshold_sweep.csv"
config_path = OOD_DIR / "okra_oe_knn_reliability_config.json"
sweep.to_csv(sweep_path, index=False)
config_path.write_text(json.dumps(config, indent=2))

# Save full calibration/validation scores for later diagnosis.
combined_scores = pd.concat([
    pd.DataFrame({
        "image_path": supported_paths,
        "split": "supported_validation",
        "label": [CLASS_NAMES[i] for i in supported_labels],
        "confidence": supported_conf,
        "knn_score": supported_knn,
    }),
    pd.DataFrame({
        "image_path": plantdoc_cal_paths,
        "split": "plantdoc_calibration",
        "label": None,
        "confidence": plantdoc_cal_conf,
        "knn_score": plantdoc_cal_knn,
    }),
    pd.DataFrame({
        "image_path": plantdoc_val_paths,
        "split": "plantdoc_validation",
        "label": None,
        "confidence": plantdoc_val_conf,
        "knn_score": plantdoc_val_knn,
    }),
], ignore_index=True)
combined_scores.to_csv(OOD_DIR / "okra_oe_knn_calibration_scores.csv", index=False)

print(f"K: {K}")
print(f"Selected confidence threshold: {conf_threshold:.6f}")
print(f"Selected kNN threshold: {knn_threshold:.6f}")
print(f"Supported validation acceptance: {supported_accepted.mean():.1%}")
print(f"PlantDoc calibration rejection: {plantdoc_cal_rejected.mean():.1%}")
print(f"PlantDoc validation diagnostic rejection: {plantdoc_val_rejected.mean():.1%}")

print("\nSupported validation acceptance by class:")
display(pd.DataFrame([
    {
        "class": class_name,
        "images": int((supported_labels == i).sum()),
        "accepted": int(supported_accepted[supported_labels == i].sum()),
        "acceptance_rate": per_class_acceptance[class_name],
    }
    for i, class_name in enumerate(CLASS_NAMES)
]).set_index("class"))

print("\nBest calibration trade-offs:")
display(sweep.head(10).round(4))

print("\nSaved threshold sweep:", sweep_path)
print("Saved reliability config:", config_path)

K: 10
Selected confidence threshold: 0.391693
Selected kNN threshold: 0.966280
Supported validation acceptance: 97.9%
PlantDoc calibration rejection: 100.0%
PlantDoc validation diagnostic rejection: 99.7%

Supported validation acceptance by class:


,images,accepted,acceptance_rate
class,,,
Healthy_Leaf,104,102,0.980769
Insects_Eaten_Leaf,86,86,1.000000
Leaf_Curl,52,48,0.923077
Leaf_Spot,45,43,0.955556
Yellow_Vein_Mosaic_Virus,133,132,0.992481



Best calibration trade-offs:


,confidence_threshold,knn_threshold,supported_acceptance,plantdoc_calibration_rejection,minimum_class_acceptance,accept_Healthy_Leaf,accept_Insects_Eaten_Leaf,accept_Leaf_Curl,accept_Leaf_Spot,accept_Yellow_Vein_Mosaic_Virus
0,0.3917,0.9663,0.9786,1.0,0.9231,0.9808,1.0000,0.9231,0.9556,0.9925
1,0.3917,0.9668,0.9786,1.0,0.9231,0.9808,1.0000,0.9231,0.9556,0.9925
2,0.3917,0.9673,0.9786,1.0,0.9231,0.9808,1.0000,0.9231,0.9556,0.9925
3,0.3917,0.9677,0.9762,1.0,0.9231,0.9808,0.9884,0.9231,0.9556,0.9925
4,0.3917,0.9678,0.9762,1.0,0.9231,0.9808,0.9884,0.9231,0.9556,0.9925
5,0.3917,0.9679,0.9762,1.0,0.9231,0.9808,0.9884,0.9231,0.9556,0.9925
6,0.3917,0.9680,0.9762,1.0,0.9231,0.9808,0.9884,0.9231,0.9556,0.9925
7,0.3917,0.9681,0.9762,1.0,0.9231,0.9808,0.9884,0.9231,0.9556,0.9925
8,0.4040,0.9663,0.9762,1.0,0.9231,0.9808,0.9884,0.9231,0.9556,0.9925
9,0.4040,0.9668,0.9762,1.0,0.9231,0.9808,0.9884,0.9231,0.9556,0.9925



Saved threshold sweep: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/ood_data/okra_oe_knn_threshold_sweep.csv
Saved reliability config: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/ood_data/okra_oe_knn_reliability_config.json


In [19]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

OOD_DIR = EXPERIMENT_DIR / "ood_data"
OE_BEST_PATH = OOD_DIR / "okra_oe_best.pth"
CONFIG_PATH = OOD_DIR / "okra_oe_knn_reliability_config.json"

with open(CONFIG_PATH) as f:
    reliability_config = json.load(f)

CONF_THRESHOLD = float(reliability_config["confidence_threshold"])
KNN_THRESHOLD = float(reliability_config["knn_threshold"])
K = int(reliability_config["k"])

oe_checkpoint = torch.load(OE_BEST_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(oe_checkpoint["model_state_dict"], strict=True)
model.to(DEVICE).eval()

ood_test_df = load_ood_split("test")
ood_test_loader = DataLoader(
    OkraOODDataset(ood_test_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

reference = F.normalize(
    features["reference_embeddings"].float(), dim=1
).to(DEVICE)

def classify_and_score(full, tiles):
    full_features = model.encode_view(full)

    batch_size, tile_count, channels, height, width = tiles.shape
    flat_tiles = tiles.reshape(batch_size * tile_count, channels, height, width)
    tile_features = model.encode_view(flat_tiles).reshape(batch_size, tile_count, -1)

    tile_weights = torch.softmax(
        model.tile_score(tile_features).squeeze(-1), dim=1
    )
    weighted_tiles = (tile_features * tile_weights.unsqueeze(-1)).sum(dim=1)
    strongest_tile = tile_features.max(dim=1).values

    logits = model.classifier(
        torch.cat([full_features, weighted_tiles, strongest_tile], dim=1)
    )
    probabilities = torch.softmax(logits.float(), dim=1)
    confidence, prediction = probabilities.max(dim=1)

    embedding = F.normalize(full_features.float(), dim=1)
    similarities = embedding @ reference.T
    knn_score = similarities.topk(K, dim=1).values.mean(dim=1)

    return prediction, confidence, knn_score

# Supported Okra internal test
test_true, test_pred, test_conf, test_knn = [], [], [], []
with torch.inference_mode():
    for full, tiles, targets in tqdm(test_loader, desc="Okra internal test"):
        full = full.to(DEVICE, non_blocking=True)
        tiles = tiles.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            pred, conf, knn = classify_and_score(full, tiles)

        test_true.extend(targets.tolist())
        test_pred.extend(pred.cpu().tolist())
        test_conf.extend(conf.cpu().tolist())
        test_knn.extend(knn.cpu().tolist())

test_true = np.asarray(test_true, dtype=np.int64)
test_pred = np.asarray(test_pred, dtype=np.int64)
test_conf = np.asarray(test_conf, dtype=np.float32)
test_knn = np.asarray(test_knn, dtype=np.float32)

accepted = (test_conf >= CONF_THRESHOLD) & (test_knn >= KNN_THRESHOLD)
test_report = classification_report(
    test_true,
    test_pred,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
test_cm = confusion_matrix(
    test_true, test_pred, labels=list(range(len(CLASS_NAMES)))
)

accepted_accuracy = (
    float(accuracy_score(test_true[accepted], test_pred[accepted]))
    if accepted.any() else None
)
accepted_macro_f1 = (
    float(f1_score(
        test_true[accepted],
        test_pred[accepted],
        labels=list(range(len(CLASS_NAMES))),
        average="macro",
        zero_division=0,
    ))
    if accepted.any() else None
)

support_metrics = {
    "checkpoint_epoch": int(oe_checkpoint["epoch"]),
    "images": int(len(test_true)),
    "accuracy_before_rejection": float(accuracy_score(test_true, test_pred)),
    "balanced_accuracy_before_rejection": float(
        balanced_accuracy_score(test_true, test_pred)
    ),
    "macro_f1_before_rejection": float(
        f1_score(
            test_true, test_pred,
            labels=list(range(len(CLASS_NAMES))),
            average="macro",
            zero_division=0,
        )
    ),
    "accepted_images": int(accepted.sum()),
    "accepted_rate": float(accepted.mean()),
    "accuracy_among_accepted": accepted_accuracy,
    "macro_f1_among_accepted": accepted_macro_f1,
    "confidence_threshold": CONF_THRESHOLD,
    "knn_threshold": KNN_THRESHOLD,
    "k": K,
}

per_class_acceptance = []
for i, class_name in enumerate(CLASS_NAMES):
    mask = test_true == i
    per_class_acceptance.append({
        "class": class_name,
        "images": int(mask.sum()),
        "accepted": int(accepted[mask].sum()),
        "accepted_rate": float(accepted[mask].mean()) if mask.any() else None,
        "accuracy_before_rejection": float(
            accuracy_score(test_true[mask], test_pred[mask])
        ) if mask.any() else None,
    })

# Held-out PlantDoc OOD test
ood_conf, ood_knn, ood_paths = [], [], []
with torch.inference_mode():
    for full, tiles in tqdm(ood_test_loader, desc="Held-out PlantDoc OOD test"):
        full = full.to(DEVICE, non_blocking=True)
        tiles = tiles.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            _, conf, knn = classify_and_score(full, tiles)

        ood_conf.extend(conf.cpu().tolist())
        ood_knn.extend(knn.cpu().tolist())

ood_conf = np.asarray(ood_conf, dtype=np.float32)
ood_knn = np.asarray(ood_knn, dtype=np.float32)
ood_paths = ood_test_df["image_path"].astype(str).tolist()

ood_accepted = (ood_conf >= CONF_THRESHOLD) & (ood_knn >= KNN_THRESHOLD)
ood_rejected = ~ood_accepted

ood_predictions = pd.DataFrame({
    "image_path": ood_paths,
    "plantdoc_class": ood_test_df["plantdoc_class"].astype(str).tolist(),
    "confidence": ood_conf,
    "knn_score": ood_knn,
    "accepted_as_okra": ood_accepted,
    "rejected_as_ood": ood_rejected,
})

ood_by_class = (
    ood_predictions.groupby("plantdoc_class")
    .agg(images=("image_path", "size"), rejected=("rejected_as_ood", "sum"))
    .reset_index()
)
ood_by_class["rejection_rate"] = ood_by_class["rejected"] / ood_by_class["images"]

# Save final reports.
report_df = pd.DataFrame(test_report).transpose()
cm_df = pd.DataFrame(test_cm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_df.index.name = "true_label"
cm_df.columns.name = "predicted_label"

support_predictions = test_df.reset_index(drop=True).copy()
support_predictions["true_label"] = [CLASS_NAMES[i] for i in test_true]
support_predictions["predicted_label"] = [CLASS_NAMES[i] for i in test_pred]
support_predictions["confidence"] = test_conf
support_predictions["knn_score"] = test_knn
support_predictions["accepted"] = accepted
support_predictions["correct"] = support_predictions["true_label"] == support_predictions["predicted_label"]

summary = {
    "frozen_policy": reliability_config,
    "okra_internal_test": support_metrics,
    "okra_test_acceptance_by_class": per_class_acceptance,
    "plantdoc_ood_test": {
        "images": int(len(ood_test_df)),
        "rejected": int(ood_rejected.sum()),
        "rejection_rate": float(ood_rejected.mean()),
    },
}

with open(OOD_DIR / "okra_oe_knn_final_test_metrics.json", "w") as f:
    json.dump(summary, f, indent=2)
report_df.to_csv(OOD_DIR / "okra_oe_internal_test_classification_report.csv")
cm_df.to_csv(OOD_DIR / "okra_oe_internal_test_confusion_matrix.csv")
support_predictions.to_csv(OOD_DIR / "okra_oe_internal_test_predictions.csv", index=False)
ood_predictions.to_csv(OOD_DIR / "okra_plantdoc_ood_test_predictions.csv", index=False)
ood_by_class.to_csv(OOD_DIR / "okra_plantdoc_ood_test_by_class.csv", index=False)

print("Frozen confidence threshold:", CONF_THRESHOLD)
print("Frozen kNN threshold:", KNN_THRESHOLD, "| K =", K)
print("\nOkra internal test:")
print(f"Images: {support_metrics['images']}")
print(f"Accuracy before rejection: {support_metrics['accuracy_before_rejection']:.4f}")
print(f"Balanced accuracy: {support_metrics['balanced_accuracy_before_rejection']:.4f}")
print(f"Macro-F1 before rejection: {support_metrics['macro_f1_before_rejection']:.4f}")
print(f"Accepted: {support_metrics['accepted_images']}/{support_metrics['images']} "
      f"({support_metrics['accepted_rate']:.1%})")
print(f"Accuracy among accepted: {support_metrics['accuracy_among_accepted']:.4f}")
print(f"Macro-F1 among accepted: {support_metrics['macro_f1_among_accepted']:.4f}")

print("\nClassification report:")
display(report_df.round(4))
print("\nConfusion matrix (true rows, predicted columns):")
display(cm_df)
print("\nOkra acceptance by class:")
display(pd.DataFrame(per_class_acceptance).set_index("class").round(4))

print("\nHeld-out PlantDoc OOD test:")
print(f"Rejected: {int(ood_rejected.sum())}/{len(ood_rejected)} ({ood_rejected.mean():.1%})")
display(ood_by_class.sort_values("rejection_rate", ascending=False).round(4))

print("\nFinal test reports saved under:", OOD_DIR)

Okra internal test:   0%|          | 0/105 [00:00<?, ?it/s]

Held-out PlantDoc OOD test:   0%|          | 0/57 [00:00<?, ?it/s]

Frozen confidence threshold: 0.3916929066181183
Frozen kNN threshold: 0.9662803423404693 | K = 10

Okra internal test:
Images: 420
Accuracy before rejection: 0.9524
Balanced accuracy: 0.9397
Macro-F1 before rejection: 0.9431
Accepted: 414/420 (98.6%)
Accuracy among accepted: 0.9589
Macro-F1 among accepted: 0.9523

Classification report:


,precision,recall,f1-score,support
Healthy_Leaf,0.9035,0.9904,0.9450,104.0000
Insects_Eaten_Leaf,0.9487,0.8605,0.9024,86.0000
Leaf_Curl,0.9623,0.9808,0.9714,52.0000
Leaf_Spot,0.9286,0.8667,0.8966,45.0000
Yellow_Vein_Mosaic_Virus,1.0000,1.0000,1.0000,133.0000
accuracy,0.9524,0.9524,0.9524,0.9524
macro avg,0.9486,0.9397,0.9431,420.0000
weighted avg,0.9533,0.9524,0.9518,420.0000



Confusion matrix (true rows, predicted columns):


...


Okra acceptance by class:


,images,accepted,accepted_rate,accuracy_before_rejection
class,,,,
Healthy_Leaf,104,102,0.9808,0.9904
Insects_Eaten_Leaf,86,84,0.9767,0.8605
Leaf_Curl,52,51,0.9808,0.9808
Leaf_Spot,45,44,0.9778,0.8667
Yellow_Vein_Mosaic_Virus,133,133,1.0000,1.0000



Held-out PlantDoc OOD test:
Rejected: 227/228 (99.6%)


,plantdoc_class,images,rejected,rejection_rate
0,Apple Scab Leaf,10,10,1.0
1,Apple leaf,9,9,1.0
2,Apple rust leaf,10,10,1.0
3,Bell_pepper leaf,8,8,1.0
4,Bell_pepper leaf spot,9,9,1.0
5,Blueberry leaf,11,11,1.0
6,Cherry leaf,10,10,1.0
7,Corn Gray leaf spot,3,3,1.0
8,Corn leaf blight,10,10,1.0
9,Corn rust leaf,10,10,1.0



Final test reports saved under: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/ood_data


In [31]:
from pathlib import Path
import json
import shutil
import importlib.util
import sys

import torch

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
OOD_DIR = EXPERIMENT_DIR / "ood_data"
PACKAGE_DIR = EXPERIMENT_DIR / "hf_package" / "okra" / "v1"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

OE_BEST_PATH = OOD_DIR / "okra_oe_best.pth"
FEATURES_PATH = OOD_DIR / "okra_oe_knn_calibration_embeddings.pt"
RELIABILITY_PATH = OOD_DIR / "okra_oe_knn_reliability_config.json"
METRICS_PATH = OOD_DIR / "okra_oe_knn_final_test_metrics.json"

for required_file in (OE_BEST_PATH, FEATURES_PATH, RELIABILITY_PATH, METRICS_PATH):
    if not required_file.exists():
        raise FileNotFoundError(required_file)

with open(RELIABILITY_PATH) as f:
    reliability = json.load(f)
with open(METRICS_PATH) as f:
    final_metrics = json.load(f)

# Copy the selected OE checkpoint.
shutil.copy2(OE_BEST_PATH, PACKAGE_DIR / "okra_oe_best.pth")

# Package only the reference bank data needed during inference.
feature_data = torch.load(FEATURES_PATH, map_location="cpu", weights_only=False)
torch.save(
    {
        "embeddings": feature_data["reference_embeddings"].float().cpu(),
        "image_paths": feature_data["reference_paths"],
        "count": len(feature_data["reference_paths"]),
        "dimension": int(feature_data["reference_embeddings"].shape[1]),
        "checkpoint": str(OE_BEST_PATH),
    },
    PACKAGE_DIR / "okra_knn_reference_bank.pt",
)

classes_json = {
    "crop": "okra",
    "model_version": "1.0.0",
    "classes": CLASS_NAMES,
}
(PACKAGE_DIR / "classes.json").write_text(json.dumps(classes_json, indent=2))

config_json = {
    "crop": "okra",
    "model_version": "1.0.0",
    "architecture": "OkraResNet50Attention",
    "classes": CLASS_NAMES,
    "input": {
        "color_mode": "RGB",
        "full_image_size": [448, 448],
        "tile_size": [288, 288],
        "tile_count": 4,
        "tile_layout": "four overlapping corners",
        "padding_color_rgb": [127, 127, 127],
        "normalization": {
            "mean": list(IMAGENET_MEAN),
            "std": list(IMAGENET_STD),
        },
    },
    "reliability": {
        "confidence_threshold": reliability["confidence_threshold"],
        "knn_threshold": reliability["knn_threshold"],
        "k": reliability["k"],
        "policy": reliability["policy"],
    },
    "training_checkpoint_epoch": int(oe_checkpoint["epoch"]),
    "final_test_metrics": final_metrics,
}
(PACKAGE_DIR / "config.json").write_text(json.dumps(config_json, indent=2))
shutil.copy2(RELIABILITY_PATH, PACKAGE_DIR / "okra_reliability_config.json")

# Self-contained model definition matching the trained architecture.
model_source = r'''
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50

class SpatialSelfAttention(nn.Module):
    def __init__(self, channels=256, num_heads=8):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.attention = nn.MultiheadAttention(
            embed_dim=channels, num_heads=num_heads, batch_first=True
        )
        self.gate = nn.Conv2d(channels * 2, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, height, width = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        normalized = self.norm(tokens)
        attended, _ = self.attention(
            normalized, normalized, normalized, need_weights=False
        )
        attended = attended.transpose(1, 2).reshape(
            batch, channels, height, width
        )
        gate = torch.sigmoid(self.gate(torch.cat([x, attended], dim=1)))
        return x + gate * attended

class OkraResNet50Attention(nn.Module):
    def __init__(self, num_classes=5, pretrained=False):
        super().__init__()
        # Inference loads trained weights; it should not download ImageNet weights.
        backbone = resnet50(weights=None)
        self.stem = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.project2 = nn.Conv2d(512, 256, kernel_size=1)
        self.project3 = nn.Conv2d(1024, 256, kernel_size=1)
        self.project4 = nn.Conv2d(2048, 256, kernel_size=1)
        self.fusion = nn.Sequential(
            nn.Conv2d(768, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.self_attention = SpatialSelfAttention(256, num_heads=8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.tile_score = nn.Linear(256, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.35),
            nn.Linear(768, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(512, num_classes),
        )

    def encode_view(self, images):
        x = self.stem(images)
        x = self.layer1(x)
        fine = self.layer2(x)
        middle = self.layer3(fine)
        deep = self.layer4(middle)

        output_size = fine.shape[-2:]
        fine = self.project2(fine)
        middle = F.interpolate(
            self.project3(middle), size=output_size,
            mode="bilinear", align_corners=False
        )
        deep = F.interpolate(
            self.project4(deep), size=output_size,
            mode="bilinear", align_corners=False
        )
        fused = self.fusion(torch.cat([fine, middle, deep], dim=1))
        attended = self.self_attention(fused)
        return self.pool(attended).flatten(1)

    def forward_with_features(self, full_images, tile_images):
        full_features = self.encode_view(full_images)
        batch_size, tile_count, channels, height, width = tile_images.shape
        flat_tiles = tile_images.reshape(
            batch_size * tile_count, channels, height, width
        )
        tile_features = self.encode_view(flat_tiles).reshape(
            batch_size, tile_count, -1
        )
        tile_weights = torch.softmax(
            self.tile_score(tile_features).squeeze(-1), dim=1
        )
        weighted_tiles = (
            tile_features * tile_weights.unsqueeze(-1)
        ).sum(dim=1)
        strongest_tile = tile_features.max(dim=1).values
        combined = torch.cat(
            [full_features, weighted_tiles, strongest_tile], dim=1
        )
        return self.classifier(combined), full_features

    def forward(self, full_images, tile_images):
        logits, _ = self.forward_with_features(full_images, tile_images)
        return logits
'''
(PACKAGE_DIR / "model.py").write_text(model_source.strip() + "\n")

inference_source = r'''
import json
from pathlib import Path

import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from torchvision.transforms import functional as TF

from model import OkraResNet50Attention

class OkraDiseasePredictor:
    def __init__(self, model_dir, device=None):
        self.model_dir = Path(model_dir)
        self.device = torch.device(
            device or ("cuda" if torch.cuda.is_available() else "cpu")
        )
        with open(self.model_dir / "config.json") as f:
            self.config = json.load(f)
        with open(self.model_dir / "classes.json") as f:
            self.class_names = json.load(f)["classes"]

        self.size = self.config["input"]["full_image_size"][0]
        self.tile_size = self.config["input"]["tile_size"][0]
        self.mean = self.config["input"]["normalization"]["mean"]
        self.std = self.config["input"]["normalization"]["std"]
        self.conf_threshold = float(
            self.config["reliability"]["confidence_threshold"]
        )
        self.knn_threshold = float(self.config["reliability"]["knn_threshold"])
        self.k = int(self.config["reliability"]["k"])

        self.model = OkraResNet50Attention(
            num_classes=len(self.class_names), pretrained=False
        )
        checkpoint = torch.load(
            self.model_dir / "okra_oe_best.pth",
            map_location=self.device,
            weights_only=False,
        )
        self.model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        self.model.to(self.device).eval()

        bank = torch.load(
            self.model_dir / "okra_knn_reference_bank.pt",
            map_location="cpu",
            weights_only=False,
        )
        self.reference = F.normalize(
            bank["embeddings"].float(), dim=1
        ).to(self.device)

    def _prepare(self, image):
        image = ImageOps.exif_transpose(image).convert("RGB")
        image = ImageOps.pad(
            image,
            (self.size, self.size),
            method=Image.Resampling.LANCZOS,
            color=tuple(self.config["input"]["padding_color_rgb"]),
        )
        full = TF.normalize(TF.to_tensor(image), self.mean, self.std)

        starts = [
            (0, 0),
            (self.size - self.tile_size, 0),
            (0, self.size - self.tile_size),
            (self.size - self.tile_size, self.size - self.tile_size),
        ]
        tiles = []
        for left, top in starts:
            tile = image.crop(
                (left, top, left + self.tile_size, top + self.tile_size)
            )
            tiles.append(TF.normalize(TF.to_tensor(tile), self.mean, self.std))
        return full.unsqueeze(0).to(self.device), torch.stack(tiles).unsqueeze(0).to(self.device)

    @torch.inference_mode()
    def predict(self, image):
        if isinstance(image, (str, Path)):
            with Image.open(image) as source:
                full, tiles = self._prepare(source)
        else:
            full, tiles = self._prepare(image)

        logits, embedding = self.model.forward_with_features(full, tiles)
        probabilities = torch.softmax(logits.float(), dim=1)[0]
        confidence, predicted_index = probabilities.max(dim=0)

        normalized_embedding = F.normalize(embedding.float(), dim=1)
        similarities = normalized_embedding @ self.reference.T
        knn_score = similarities.topk(
            min(self.k, self.reference.shape[0]), dim=1
        ).values.mean().item()

        confidence = float(confidence.item())
        confidence_ok = confidence >= self.conf_threshold
        knn_ok = knn_score >= self.knn_threshold
        accepted = confidence_ok and knn_ok

        top_values, top_indices = probabilities.topk(
            min(3, len(self.class_names))
        )
        top = [
            {
                "class": self.class_names[int(index)],
                "confidence": float(value),
            }
            for value, index in zip(top_values.tolist(), top_indices.tolist())
        ]

        reasons = []
        if not confidence_ok:
            reasons.append("LOW_CONFIDENCE")
        if not knn_ok:
            reasons.append("LOW_OKRA_SIMILARITY")

        return {
            "crop": "okra",
            "model_version": self.config["model_version"],
            "decision": "PREDICTION" if accepted else "REJECTED",
            "predicted_class": (
                self.class_names[int(predicted_index)] if accepted else None
            ),
            "confidence": confidence,
            "knn_score": float(knn_score),
            "confidence_threshold": self.conf_threshold,
            "knn_threshold": self.knn_threshold,
            "top3": top,
            "reason_codes": reasons,
        }
'''
(PACKAGE_DIR / "inference.py").write_text(inference_source.strip() + "\n")

(PACKAGE_DIR / "requirements.txt").write_text(
    "torch\n"
    "torchvision\n"
    "Pillow\n"
)
(PACKAGE_DIR / "README.md").write_text(
    f"""# Okra Disease Classifier

Five-class Okra leaf classifier with outlier-exposure fine-tuning and a
confidence + kNN rejection policy.

## Classes
{chr(10).join(f"- {name}" for name in CLASS_NAMES)}

## Final evaluation
- Okra internal test: accuracy {final_metrics['okra_internal_test']['accuracy_before_rejection']:.4f},
  macro-F1 {final_metrics['okra_internal_test']['macro_f1_before_rejection']:.4f}
- Accepted Okra test images: {final_metrics['okra_internal_test']['accepted_rate']:.1%}
- Accuracy among accepted Okra test images:
  {final_metrics['okra_internal_test']['accuracy_among_accepted']:.4f}
- Held-out PlantDoc rejection:
  {final_metrics['plantdoc_ood_test']['rejection_rate']:.1%}
- Rejection evaluation uses the listed held-out PlantDoc test set; performance
  on arbitrary real-world inputs may differ.

## Inference
```python
from inference import OkraDiseasePredictor

predictor = OkraDiseasePredictor("path/to/okra/v1")
result = predictor.predict("leaf.jpg")
print(result)

_IncompleteInputError: incomplete input (3361628640.py, line 331)

In [30]:
readme_lines = [
    "# Okra Disease Classifier",
    "",
    "Five-class Okra leaf classifier with outlier-exposure fine-tuning",
    "and a confidence plus kNN rejection policy.",
    "",
    "Classes:",
    *[f"- {name}" for name in CLASS_NAMES],
    "",
    "Final evaluation:",
    f"- Okra internal test accuracy: "
    f"{final_metrics['okra_internal_test']['accuracy_before_rejection']:.4f}",
    f"- Okra internal test macro-F1: "
    f"{final_metrics['okra_internal_test']['macro_f1_before_rejection']:.4f}",
    f"- Accepted Okra test images: "
    f"{final_metrics['okra_internal_test']['accepted_rate']:.1%}",
    f"- Accuracy among accepted Okra images: "
    f"{final_metrics['okra_internal_test']['accuracy_among_accepted']:.4f}",
    f"- Held-out PlantDoc rejection: "
    f"{final_metrics['plantdoc_ood_test']['rejection_rate']:.1%}",
    "",
    "Inference example:",
    "from inference import OkraDiseasePredictor",
    'predictor = OkraDiseasePredictor("path/to/okra/v1")',
    'result = predictor.predict("leaf.jpg")',
    "print(result)",
    "",
    "PlantDoc results describe the held-out PlantDoc test set;",
    "performance on other real-world inputs may differ.",
]
(PACKAGE_DIR / "README.md").write_text("\n".join(readme_lines) + "\n")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/final_year_project/okra/okra_model_experiment/hf_package/okra/v1/README.md'

In [29]:
import json
from pathlib import Path

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
OOD_DIR = EXPERIMENT_DIR / "ood_data"
PACKAGE_DIR = EXPERIMENT_DIR / "hf_package" / "okra" / "v1"

with open(OOD_DIR / "okra_oe_knn_final_test_metrics.json") as f:
    final_metrics = json.load(f)

with open(PACKAGE_DIR / "classes.json") as f:
    class_names = json.load(f)["classes"]

okra_test = final_metrics["okra_internal_test"]
plantdoc_test = final_metrics["plantdoc_ood_test"]

readme_lines = [
    "# Okra Disease Classifier",
    "",
    "Five-class Okra leaf classifier with outlier-exposure fine-tuning",
    "and a confidence plus kNN rejection policy.",
    "",
    "Classes:",
    *[f"- {name}" for name in class_names],
    "",
    "Final evaluation:",
    f"- Okra internal test accuracy: {okra_test['accuracy_before_rejection']:.4f}",
    f"- Okra internal test macro-F1: {okra_test['macro_f1_before_rejection']:.4f}",
    f"- Accepted Okra test images: {okra_test['accepted_rate']:.1%}",
    f"- Accuracy among accepted Okra images: {okra_test['accuracy_among_accepted']:.4f}",
    f"- Held-out PlantDoc rejection: {plantdoc_test['rejection_rate']:.1%}",
    "",
    "Inference example:",
    "from inference import OkraDiseasePredictor",
    'predictor = OkraDiseasePredictor("path/to/okra/v1")',
    'result = predictor.predict("leaf.jpg")',
    "print(result)",
    "",
    "PlantDoc results describe the held-out PlantDoc test set;",
    "performance on other real-world inputs may differ.",
]

(PACKAGE_DIR / "README.md").write_text("\n".join(readme_lines) + "\n")
print("README saved:", PACKAGE_DIR / "README.md")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/final_year_project/okra/okra_model_experiment/hf_package/okra/v1/classes.json'

In [32]:
from pathlib import Path
import json
import sys
import torch

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/okra_model_experiment"
)
OOD_DIR = EXPERIMENT_DIR / "ood_data"
PACKAGE_DIR = EXPERIMENT_DIR / "hf_package" / "okra" / "v1"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    "Healthy_Leaf",
    "Insects_Eaten_Leaf",
    "Leaf_Curl",
    "Leaf_Spot",
    "Yellow_Vein_Mosaic_Virus",
]

CHECKPOINT_PATH = OOD_DIR / "okra_oe_best.pth"
FEATURES_PATH = OOD_DIR / "okra_oe_knn_calibration_embeddings.pt"
RELIABILITY_PATH = OOD_DIR / "okra_oe_knn_reliability_config.json"
METRICS_PATH = OOD_DIR / "okra_oe_knn_final_test_metrics.json"

for path in [CHECKPOINT_PATH, FEATURES_PATH, RELIABILITY_PATH, METRICS_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)

with open(RELIABILITY_PATH) as f:
    reliability = json.load(f)
with open(METRICS_PATH) as f:
    final_metrics = json.load(f)

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
feature_data = torch.load(FEATURES_PATH, map_location="cpu", weights_only=False)

# Save class list, reliability settings, model checkpoint, and kNN reference bank.
(PACKAGE_DIR / "classes.json").write_text(json.dumps({
    "crop": "okra",
    "model_version": "1.0.0",
    "classes": CLASS_NAMES,
}, indent=2))

config = {
    "crop": "okra",
    "model_version": "1.0.0",
    "architecture": "OkraResNet50Attention",
    "classes": CLASS_NAMES,
    "input": {
        "color_mode": "RGB",
        "full_image_size": [448, 448],
        "tile_size": [288, 288],
        "tile_count": 4,
        "tile_layout": "four overlapping corners",
        "padding_color_rgb": [127, 127, 127],
        "normalization": {
            "mean": [0.485, 0.456, 0.406],
            "std": [0.229, 0.224, 0.225],
        },
    },
    "reliability": {
        "confidence_threshold": reliability["confidence_threshold"],
        "knn_threshold": reliability["knn_threshold"],
        "k": reliability["k"],
        "policy": reliability["policy"],
    },
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "final_test_metrics": final_metrics,
}
(PACKAGE_DIR / "config.json").write_text(json.dumps(config, indent=2))
(PACKAGE_DIR / "okra_reliability_config.json").write_text(
    json.dumps(reliability, indent=2)
)
torch.save(checkpoint, PACKAGE_DIR / "okra_oe_best.pth")

torch.save({
    "embeddings": feature_data["reference_embeddings"].float().cpu(),
    "image_paths": feature_data["reference_paths"],
    "count": len(feature_data["reference_paths"]),
    "dimension": int(feature_data["reference_embeddings"].shape[1]),
}, PACKAGE_DIR / "okra_knn_reference_bank.pt")

model_source = r'''
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50

class SpatialSelfAttention(nn.Module):
    def __init__(self, channels=256, num_heads=8):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.attention = nn.MultiheadAttention(
            embed_dim=channels, num_heads=num_heads, batch_first=True
        )
        self.gate = nn.Conv2d(channels * 2, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, height, width = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        normalized = self.norm(tokens)
        attended, _ = self.attention(
            normalized, normalized, normalized, need_weights=False
        )
        attended = attended.transpose(1, 2).reshape(
            batch, channels, height, width
        )
        gate = torch.sigmoid(self.gate(torch.cat([x, attended], dim=1)))
        return x + gate * attended

class OkraResNet50Attention(nn.Module):
    def __init__(self, num_classes=5, pretrained=False):
        super().__init__()
        backbone = resnet50(weights=None)
        self.stem = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.project2 = nn.Conv2d(512, 256, kernel_size=1)
        self.project3 = nn.Conv2d(1024, 256, kernel_size=1)
        self.project4 = nn.Conv2d(2048, 256, kernel_size=1)
        self.fusion = nn.Sequential(
            nn.Conv2d(768, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.self_attention = SpatialSelfAttention(256, num_heads=8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.tile_score = nn.Linear(256, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.35),
            nn.Linear(768, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(512, num_classes),
        )

    def encode_view(self, images):
        x = self.stem(images)
        x = self.layer1(x)
        fine = self.layer2(x)
        middle = self.layer3(fine)
        deep = self.layer4(middle)
        output_size = fine.shape[-2:]
        fine = self.project2(fine)
        middle = F.interpolate(
            self.project3(middle), size=output_size,
            mode="bilinear", align_corners=False
        )
        deep = F.interpolate(
            self.project4(deep), size=output_size,
            mode="bilinear", align_corners=False
        )
        fused = self.fusion(torch.cat([fine, middle, deep], dim=1))
        attended = self.self_attention(fused)
        return self.pool(attended).flatten(1)

    def forward_with_features(self, full_images, tile_images):
        full_features = self.encode_view(full_images)
        batch_size, tile_count, channels, height, width = tile_images.shape
        flat_tiles = tile_images.reshape(
            batch_size * tile_count, channels, height, width
        )
        tile_features = self.encode_view(flat_tiles).reshape(
            batch_size, tile_count, -1
        )
        tile_weights = torch.softmax(
            self.tile_score(tile_features).squeeze(-1), dim=1
        )
        weighted_tiles = (
            tile_features * tile_weights.unsqueeze(-1)
        ).sum(dim=1)
        strongest_tile = tile_features.max(dim=1).values
        combined = torch.cat(
            [full_features, weighted_tiles, strongest_tile], dim=1
        )
        return self.classifier(combined), full_features

    def forward(self, full_images, tile_images):
        logits, _ = self.forward_with_features(full_images, tile_images)
        return logits
'''
(PACKAGE_DIR / "model.py").write_text(model_source.strip() + "\n")

inference_source = r'''
import json
from pathlib import Path

import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from torchvision.transforms import functional as TF

from model import OkraResNet50Attention

class OkraDiseasePredictor:
    def __init__(self, model_dir, device=None):
        self.model_dir = Path(model_dir)
        self.device = torch.device(
            device or ("cuda" if torch.cuda.is_available() else "cpu")
        )
        with open(self.model_dir / "config.json") as f:
            self.config = json.load(f)
        with open(self.model_dir / "classes.json") as f:
            self.classes = json.load(f)["classes"]

        inputs = self.config["input"]
        self.size = int(inputs["full_image_size"][0])
        self.tile_size = int(inputs["tile_size"][0])
        self.mean = inputs["normalization"]["mean"]
        self.std = inputs["normalization"]["std"]
        policy = self.config["reliability"]
        self.confidence_threshold = float(policy["confidence_threshold"])
        self.knn_threshold = float(policy["knn_threshold"])
        self.k = int(policy["k"])

        self.model = OkraResNet50Attention(
            num_classes=len(self.classes), pretrained=False
        )
        checkpoint = torch.load(
            self.model_dir / "okra_oe_best.pth",
            map_location=self.device,
            weights_only=False,
        )
        self.model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        self.model.to(self.device).eval()

        bank = torch.load(
            self.model_dir / "okra_knn_reference_bank.pt",
            map_location="cpu",
            weights_only=False,
        )
        self.reference = F.normalize(bank["embeddings"].float(), dim=1).to(self.device)

    def _prepare(self, image):
        image = ImageOps.exif_transpose(image).convert("RGB")
        image = ImageOps.pad(
            image,
            (self.size, self.size),
            method=Image.Resampling.LANCZOS,
            color=tuple(self.config["input"]["padding_color_rgb"]),
        )
        full = TF.normalize(TF.to_tensor(image), self.mean, self.std)
        starts = [
            (0, 0),
            (self.size - self.tile_size, 0),
            (0, self.size - self.tile_size),
            (self.size - self.tile_size, self.size - self.tile_size),
        ]
        tiles = []
        for left, top in starts:
            tile = image.crop(
                (left, top, left + self.tile_size, top + self.tile_size)
            )
            tiles.append(TF.normalize(TF.to_tensor(tile), self.mean, self.std))
        return (
            full.unsqueeze(0).to(self.device),
            torch.stack(tiles).unsqueeze(0).to(self.device),
        )

    @torch.inference_mode()
    def predict(self, image):
        if isinstance(image, (str, Path)):
            with Image.open(image) as source:
                full, tiles = self._prepare(source)
        else:
            full, tiles = self._prepare(image)

        logits, embedding = self.model.forward_with_features(full, tiles)
        probabilities = torch.softmax(logits.float(), dim=1)[0]
        confidence, predicted_index = probabilities.max(dim=0)
        normalized_embedding = F.normalize(embedding.float(), dim=1)
        similarities = normalized_embedding @ self.reference.T
        knn_score = similarities.topk(
            min(self.k, self.reference.shape[0]), dim=1
        ).values.mean().item()

        confidence = float(confidence.item())
        accepted = (
            confidence >= self.confidence_threshold
            and knn_score >= self.knn_threshold
        )
        top_values, top_indices = probabilities.topk(min(3, len(self.classes)))
        top3 = [
            {"class": self.classes[int(i)], "confidence": float(v)}
            for v, i in zip(top_values.tolist(), top_indices.tolist())
        ]

        reasons = []
        if confidence < self.confidence_threshold:
            reasons.append("LOW_CONFIDENCE")
        if knn_score < self.knn_threshold:
            reasons.append("LOW_OKRA_SIMILARITY")

        return {
            "crop": "okra",
            "model_version": self.config["model_version"],
            "decision": "PREDICTION" if accepted else "REJECTED",
            "predicted_class": self.classes[int(predicted_index)] if accepted else None,
            "confidence": confidence,
            "knn_score": float(knn_score),
            "confidence_threshold": self.confidence_threshold,
            "knn_threshold": self.knn_threshold,
            "top3": top3,
            "reason_codes": reasons,
        }
'''
(PACKAGE_DIR / "inference.py").write_text(inference_source.strip() + "\n")
(PACKAGE_DIR / "requirements.txt").write_text("torch\ntorchvision\nPillow\n")

okra_test = final_metrics["okra_internal_test"]
plantdoc_test = final_metrics["plantdoc_ood_test"]
readme_lines = [
    "# Okra Disease Classifier",
    "",
    "Five-class Okra classifier with outlier-exposure fine-tuning and a confidence plus kNN rejection policy.",
    "",
    "Classes:",
    *[f"- {name}" for name in CLASS_NAMES],
    "",
    "Final evaluation:",
    f"- Okra internal test accuracy: {okra_test['accuracy_before_rejection']:.4f}",
    f"- Okra internal test macro-F1: {okra_test['macro_f1_before_rejection']:.4f}",
    f"- Accepted Okra test images: {okra_test['accepted_rate']:.1%}",
    f"- Accuracy among accepted Okra images: {okra_test['accuracy_among_accepted']:.4f}",
    f"- Held-out PlantDoc rejection: {plantdoc_test['rejection_rate']:.1%}",
    "",
    "Inference:",
    "from inference import OkraDiseasePredictor",
    'predictor = OkraDiseasePredictor("path/to/okra/v1")',
    'result = predictor.predict("leaf.jpg")',
    "print(result)",
    "",
    "PlantDoc results describe this held-out dataset; performance on other real-world inputs may differ.",
]
(PACKAGE_DIR / "README.md").write_text("\n".join(readme_lines) + "\n")

# Smoke-test packaged inference on a training image.
train_manifest = pd.read_csv(EXPERIMENT_DIR / "okra_train_manifest.csv")
sample_image = str(train_manifest.iloc[0]["image_path"])

sys.path.insert(0, str(PACKAGE_DIR))
from inference import OkraDiseasePredictor

predictor = OkraDiseasePredictor(PACKAGE_DIR, device="cuda" if torch.cuda.is_available() else "cpu")

print("Package folder:", PACKAGE_DIR)
print("Package files:")
for path in sorted(PACKAGE_DIR.iterdir()):
    if path.is_file():
        print(f"- {path.name} ({path.stat().st_size / (1024**2):.2f} MB)")
print("\nTraining-image smoke test:")
print("Image:", sample_image)
print("Result:", predictor.predict(sample_image))

Package folder: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/hf_package/okra/v1
Package files:
- README.md (0.00 MB)
- classes.json (0.00 MB)
- config.json (0.00 MB)
- inference.py (0.00 MB)
- model.py (0.00 MB)
- okra_knn_reference_bank.pt (2.15 MB)
- okra_oe_best.pth (103.27 MB)
- okra_reliability_config.json (0.00 MB)
- requirements.txt (0.00 MB)

Training-image smoke test:
Image: /content/drive/MyDrive/final_year_project/okra/okra_model_experiment/okra_images_final/Healthy_Leaf/HL-00956.jpg
Result: {'crop': 'okra', 'model_version': '1.0.0', 'decision': 'PREDICTION', 'predicted_class': 'Healthy_Leaf', 'confidence': 0.9578863382339478, 'knn_score': 0.9837847948074341, 'confidence_threshold': 0.3916929066181183, 'knn_threshold': 0.9662803423404693, 'top3': [{'class': 'Healthy_Leaf', 'confidence': 0.9578863382339478}, {'class': 'Insects_Eaten_Leaf', 'confidence': 0.01870635338127613}, {'class': 'Leaf_Spot', 'confidence': 0.008522347547113895}], 'reason_codes': [

In [34]:
%pip -q install -U huggingface_hub

from pathlib import Path
from huggingface_hub import HfApi, notebook_login

PACKAGE_DIR = Path(
    "/content/drive/MyDrive/final_year_project/okra/"
    "okra_model_experiment/hf_package/okra/v1"
)
REPO_ID = "virajwirle/plant-disease-models"

assert PACKAGE_DIR.is_dir(), f"Package folder not found: {PACKAGE_DIR}"
required = [
    "README.md",
    "classes.json",
    "config.json",
    "inference.py",
    "model.py",
    "okra_knn_reference_bank.pt",
    "okra_oe_best.pth",
    "okra_reliability_config.json",
    "requirements.txt",
]
missing = [name for name in required if not (PACKAGE_DIR / name).is_file()]
assert not missing, f"Package is missing files: {missing}"

# Securely authenticate in Colab if no Hugging Face token is already configured.
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None

if not token:
    notebook_login()

api = HfApi(token=token) if token else HfApi()
api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True)

result = api.upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=str(PACKAGE_DIR),
    path_in_repo="okra/v1",
    commit_message="Add Okra disease classifier package v1",
)

print("Upload complete:", result.commit_url)
print("Package:", f"https://huggingface.co/{REPO_ID}/tree/main/okra/v1")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 14.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.2.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.2.0 which is incompatible.
tokenizers 0.23.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.2.0 which is incompatible.
gradio-client 2.7.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.2.0 which is incompatible.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/hf_api.py:11477: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


Upload complete: https://huggingface.co/virajwirle/plant-disease-models/commit/dcff8aa4918d5445a0784366f8ef843810c3c4da
Package: https://huggingface.co/virajwirle/plant-disease-models/tree/main/okra/v1
